In [1]:
# =============================================================================
# NPCI RETAIL PAYMENTS STATISTICS
# STEP 1: RAW DATASET AUDIT
# =============================================================================

import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)


# =============================================================================
# 1. INPUT FILE
# =============================================================================

INPUT_FILE = "NPCI-Retail-Payments-Statistics---Excel-Format.xlsx"

print("=" * 90)
print("STEP 1: RAW DATASET AUDIT")
print("=" * 90)

print("\nINPUT FILE")
print("-" * 90)
print("Input file :", INPUT_FILE)


# =============================================================================
# 2. FILE EXISTENCE CHECK
# =============================================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Place the Excel file in the current working directory."
    )

file_size = os.path.getsize(INPUT_FILE)

print("File size  :", f"{file_size:,}", "bytes")


# =============================================================================
# 3. LOAD WORKBOOK
# =============================================================================

xls = pd.ExcelFile(INPUT_FILE)

print("\n" + "=" * 90)
print("1. WORKBOOK INFORMATION")
print("=" * 90)

print("Number of sheets :", len(xls.sheet_names))
print("Sheet names      :", xls.sheet_names)


# =============================================================================
# 4. LOAD ALL SHEETS WITHOUT MODIFYING DATA
# =============================================================================

raw_sheets = {}

for sheet in xls.sheet_names:

    df = pd.read_excel(
        INPUT_FILE,
        sheet_name=sheet,
        header=None
    )

    raw_sheets[sheet] = df.copy()

    print("\n" + "-" * 90)
    print("SHEET:", sheet)
    print("-" * 90)

    print("Rows    :", df.shape[0])
    print("Columns :", df.shape[1])


# =============================================================================
# 5. SELECT PRIMARY SHEET
# =============================================================================

SHEET_NAME = xls.sheet_names[0]

raw_df = raw_sheets[SHEET_NAME].copy()

print("\n" + "=" * 90)
print("2. RAW SHEET STRUCTURE")
print("=" * 90)

print("Selected sheet :", SHEET_NAME)
print("Raw rows       :", raw_df.shape[0])
print("Raw columns    :", raw_df.shape[1])


# =============================================================================
# 6. FIRST 10 RAW ROWS
# =============================================================================

print("\n" + "=" * 90)
print("3. FIRST 10 RAW ROWS")
print("=" * 90)

display(raw_df.head(10))


# =============================================================================
# 7. LAST 10 RAW ROWS
# =============================================================================

print("\n" + "=" * 90)
print("4. LAST 10 RAW ROWS")
print("=" * 90)

display(raw_df.tail(10))


# =============================================================================
# 8. RAW COLUMN INFORMATION
# =============================================================================

print("\n" + "=" * 90)
print("5. RAW COLUMN INFORMATION")
print("=" * 90)

column_info = pd.DataFrame({
    "Column_Index": range(len(raw_df.columns)),
    "Column_Name": raw_df.columns.astype(str),
    "Non_Null_Count": raw_df.notna().sum().values,
    "Missing_Count": raw_df.isna().sum().values,
    "Unique_Count": [
        raw_df[col].nunique(dropna=True)
        for col in raw_df.columns
    ]
})

display(column_info)


# =============================================================================
# 9. CELL TYPE ANALYSIS
# =============================================================================

print("\n" + "=" * 90)
print("6. CELL TYPE ANALYSIS")
print("=" * 90)

numeric_count = 0
string_count = 0
datetime_count = 0
boolean_count = 0
other_count = 0
missing_count = 0

for col in raw_df.columns:

    for value in raw_df[col]:

        if pd.isna(value):
            missing_count += 1

        elif isinstance(value, (bool, np.bool_)):
            boolean_count += 1

        elif isinstance(value, (pd.Timestamp,)):
            datetime_count += 1

        elif isinstance(value, (int, float, np.integer, np.floating)):
            numeric_count += 1

        elif isinstance(value, str):
            string_count += 1

        else:
            other_count += 1


print("Numeric     :", numeric_count)
print("String      :", string_count)
print("Datetime    :", datetime_count)
print("Boolean     :", boolean_count)
print("Other       :", other_count)
print("Missing     :", missing_count)


# =============================================================================
# 10. DATASET-WIDE MISSINGNESS
# =============================================================================

print("\n" + "=" * 90)
print("7. DATASET-WIDE MISSINGNESS")
print("=" * 90)

total_cells = raw_df.size
total_missing = raw_df.isna().sum().sum()
total_present = total_cells - total_missing

print("Total cells     :", total_cells)
print("Missing cells   :", total_missing)
print("Present cells   :", total_present)
print(
    "Missing %       :",
    round((total_missing / total_cells) * 100, 4)
)


# =============================================================================
# 11. DATASET-WIDE ZERO COUNT
# =============================================================================

print("\n" + "=" * 90)
print("8. ZERO-VALUE ANALYSIS")
print("=" * 90)

numeric_raw = raw_df.apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

zero_mask = numeric_raw.eq(0)

zero_count = zero_mask.sum().sum()

print("Numeric zero cells :", int(zero_count))


# =============================================================================
# 12. COLUMN-WISE MISSINGNESS
# =============================================================================

print("\n" + "=" * 90)
print("9. COLUMN-WISE MISSINGNESS")
print("=" * 90)

missing_by_column = pd.DataFrame({
    "Column_Index": range(len(raw_df.columns)),
    "Column": raw_df.columns,
    "Missing_Count": raw_df.isna().sum().values,
    "Present_Count": raw_df.notna().sum().values,
    "Missing_Percentage": (
        raw_df.isna().mean().values * 100
    )
})

missing_by_column = missing_by_column.sort_values(
    "Missing_Count",
    ascending=False
)

display(missing_by_column)


# =============================================================================
# 13. ROW-WISE MISSINGNESS
# =============================================================================

print("\n" + "=" * 90)
print("10. ROW-WISE MISSINGNESS")
print("=" * 90)

row_missing = pd.DataFrame({
    "Row_Index": raw_df.index,
    "Missing_Count": raw_df.isna().sum(axis=1).values,
    "Present_Count": raw_df.notna().sum(axis=1).values,
    "Missing_Percentage": (
        raw_df.isna().mean(axis=1).values * 100
    )
})

display(row_missing)


# =============================================================================
# 14. COMPLETELY EMPTY ROWS
# =============================================================================

print("\n" + "=" * 90)
print("11. COMPLETELY EMPTY ROW CHECK")
print("=" * 90)

empty_rows = raw_df.isna().all(axis=1)

print("Completely empty rows :", int(empty_rows.sum()))

if empty_rows.sum() > 0:
    print("Empty row indices:")
    print(raw_df.index[empty_rows].tolist())


# =============================================================================
# 15. COMPLETELY EMPTY COLUMNS
# =============================================================================

print("\n" + "=" * 90)
print("12. COMPLETELY EMPTY COLUMN CHECK")
print("=" * 90)

empty_columns = raw_df.isna().all(axis=0)

print("Completely empty columns :", int(empty_columns.sum()))

if empty_columns.sum() > 0:
    print("Empty column indices:")
    print(raw_df.columns[empty_columns].tolist())


# =============================================================================
# 16. DUPLICATE ROW ANALYSIS
# =============================================================================

print("\n" + "=" * 90)
print("13. DUPLICATE ROW ANALYSIS")
print("=" * 90)

duplicate_mask = raw_df.duplicated(
    keep=False
)

duplicate_count = int(duplicate_mask.sum())

duplicate_groups = raw_df.duplicated(
    keep=False
).sum()

print("Duplicate row instances :", duplicate_count)

if duplicate_count > 0:

    print("\nDuplicate rows:")
    display(
        raw_df.loc[duplicate_mask]
    )

    print(
        "\nNumber of duplicate groups :",
        raw_df.loc[duplicate_mask].duplicated(
            keep=False
        )
    )


# =============================================================================
# 17. DATA TYPE REPORT
# =============================================================================

print("\n" + "=" * 90)
print("14. DATA TYPE REPORT")
print("=" * 90)

dtype_report = pd.DataFrame({
    "Column_Index": range(len(raw_df.columns)),
    "Column": raw_df.columns,
    "Data_Type": [
        str(dtype)
        for dtype in raw_df.dtypes
    ]
})

display(dtype_report)


# =============================================================================
# 18. UNIQUE VALUE INSPECTION
# =============================================================================

print("\n" + "=" * 90)
print("15. UNIQUE VALUE INSPECTION")
print("=" * 90)

for col in raw_df.columns:

    unique_values = raw_df[col].dropna().unique()

    print("\nColumn:", col)
    print("Unique values:", len(unique_values))

    # Display only a reasonable number
    if len(unique_values) <= 20:
        print(unique_values)
    else:
        print(unique_values[:20])
        print("...")


# =============================================================================
# 19. POSSIBLE TITLE / HEADER ROW INSPECTION
# =============================================================================

print("\n" + "=" * 90)
print("16. HEADER / STRUCTURAL ROW INSPECTION")
print("=" * 90)

for i in range(min(10, len(raw_df))):

    print("\nRow", i)
    print(
        raw_df.iloc[i]
        .dropna()
        .to_dict()
    )


# =============================================================================
# 20. RAW DATA SNAPSHOT
# =============================================================================

print("\n" + "=" * 90)
print("17. RAW DATA SNAPSHOT")
print("=" * 90)

raw_snapshot = raw_df.copy()

display(raw_snapshot)


# =============================================================================
# 21. CREATE BASELINE COPY
# =============================================================================

baseline_df = raw_df.copy(deep=True)


# =============================================================================
# 22. BASELINE METRICS
# =============================================================================

baseline_metrics = {
    "Input_File": INPUT_FILE,
    "Sheet": SHEET_NAME,
    "Raw_Rows": raw_df.shape[0],
    "Raw_Columns": raw_df.shape[1],
    "Total_Cells": raw_df.size,
    "Total_Missing_Cells": int(total_missing),
    "Total_Present_Cells": int(total_present),
    "Total_Numeric_Zero_Cells": int(zero_count),
    "Duplicate_Row_Instances": duplicate_count,
    "Completely_Empty_Rows": int(empty_rows.sum()),
    "Completely_Empty_Columns": int(empty_columns.sum()),
    "Numeric_Cells": numeric_count,
    "String_Cells": string_count,
    "Datetime_Cells": datetime_count,
    "Boolean_Cells": boolean_count,
    "Other_Cells": other_count
}

baseline_report = pd.DataFrame(
    [baseline_metrics]
)

print("\n" + "=" * 90)
print("18. RAW DATA BASELINE")
print("=" * 90)

display(baseline_report)


# =============================================================================
# 23. SAVE BASELINE REPORT
# =============================================================================

BASELINE_REPORT = "NPCI_Step1_Raw_Data_Audit_Report.xlsx"

with pd.ExcelWriter(
    BASELINE_REPORT,
    engine="openpyxl"
) as writer:

    baseline_report.to_excel(
        writer,
        sheet_name="Baseline",
        index=False
    )

    column_info.to_excel(
        writer,
        sheet_name="Column_Info",
        index=False
    )

    missing_by_column.to_excel(
        writer,
        sheet_name="Missing_By_Column",
        index=False
    )

    row_missing.to_excel(
        writer,
        sheet_name="Missing_By_Row",
        index=False
    )

    dtype_report.to_excel(
        writer,
        sheet_name="Data_Types",
        index=False
    )


# =============================================================================
# 24. FINAL STEP 1 VALIDATION
# =============================================================================

print("\n" + "=" * 90)
print("STEP 1 VALIDATION")
print("=" * 90)

assert raw_df.shape == baseline_df.shape

assert (
    raw_df.isna().sum().sum()
    ==
    baseline_df.isna().sum().sum()
)

assert (
    numeric_raw.eq(0).sum().sum()
    ==
    baseline_df.apply(
        lambda col: pd.to_numeric(
            col,
            errors="coerce"
        )
    ).eq(0).sum().sum()
)

print("✓ Original dataset loaded successfully")
print("✓ Raw dimensions recorded")
print("✓ Missing values measured")
print("✓ Zero values measured")
print("✓ Duplicate rows checked")
print("✓ Data types inspected")
print("✓ Unique values inspected")
print("✓ Baseline established")
print("✓ NO DATA WAS MODIFIED")
print("✓ NO VALUES WERE IMPUTED")
print("✓ NO NaN VALUES WERE REPLACED")
print("✓ NO ZERO VALUES WERE CHANGED")


# =============================================================================
# 25. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("STEP 1 FINAL REPORT")
print("=" * 90)

display(baseline_report)

print("\n" + "=" * 90)
print("STEP 1 COMPLETE")
print("=" * 90)

print("\nOutput report:")
print(BASELINE_REPORT)

print("\nREADY FOR STEP 2.")

STEP 1: RAW DATASET AUDIT

INPUT FILE
------------------------------------------------------------------------------------------
Input file : NPCI-Retail-Payments-Statistics---Excel-Format.xlsx
File size  : 707,627 bytes

1. WORKBOOK INFORMATION
Number of sheets : 1
Sheet names      : ['Sheet1']

------------------------------------------------------------------------------------------
SHEET: Sheet1
------------------------------------------------------------------------------------------
Rows    : 50
Columns : 42

2. RAW SHEET STRUCTURE
Selected sheet : Sheet1
Raw rows       : 50
Raw columns    : 42

3. FIRST 10 RAW ROWS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RETAIL PAYMENTS STATISTICS ON NPCI PLATFORMS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sr. No.,NPCI Operated Systems,FY22-23 Q1,NaN,FY22-23 Q2,NaN,FY22-23 Q3,NaN,FY22-23 Q4,NaN,F.Y-2022-23,NaN,F.Y-2023-24 Q1,NaN,F.Y-2023-24 Q2,NaN,F.Y-2023-24 Q3,NaN,F.Y-2023-24 Q4,NaN,F.Y-2024-25 Q1,NaN,F.Y-2024-25 Q2,NaN,F.Y-2024-25 Q3,NaN,F.Y-2024-25 Q4,NaN,F.Y-2025-26 Q1,NaN,F.Y-2025-26 Q2,NaN,F.Y-2025-26 Q3,NaN,F.Y-2025-26 Q4,NaN,2026-04-01 00:00:00,NaN,2026-05-01 00:00:00,NaN,2026-06-01 00:00:00,NaN
4,NaN,Financial Txns:,Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn),Volume (in Mn),Value (in Bn)
5,1,NFS - National Financial Switch,1013.772972,4239.737233,1000.45151,4063.901403,1015.517594,4202.804709,988.443205,4117.746292,4018.185281,16624.189636,998.149047,4257.621579,976.72966,4033.830614,985.759856,4168.783237,948.421058,4057.458564,924.526982,4029.910139,897.519953,3809.531659,903.171538,3958.733734,852.43,3820.92,804.835128,3641.815599,793.194452,3520.594486,799.725028,3653.187382,779.03,3652.92,250.74,1192.34,257.84,1208.58,246.624956,1127.487492
6,1.1,NFS - ATM Cash Withdrawal *,1013.194083,4233.880656,999.79525,4057.696968,1014.716752,4195.277034,987.380063,4108.43295,4015.086148,16595.287608,996.995667,4247.492945,975.460207,4023.037268,984.38525,4156.693693,947.015229,4044.760875,923.02043,4016.761614,895.966505,3795.897977,901.597363,3944.105241,850.71,3805.51,803.150913,3625.929087,791.348517,3505.867179,797.894925,3640.462796,777.57,3641.06,250.36,1188.8,257.46,1205.19,246.229058,1124.182163
7,NaN,ATM,965.583013,4057.281658,952.909465,3892.0861,968.193496,4025.487173,943.281486,3942.454156,3829.96746,15917.309086,950.71896,4072.218254,930.425103,3862.889547,939.116627,3989.796972,904.297644,3881.435562,886.84355,3886.092408,861.665272,3681.711944,868.18828,3829.415191,820.56,3698.65,773.712476,3520.443483,762.587166,3407.6769,769.88247,3541.378186,751.78,3543.83,241.88,1156.05,248.72,1171.59,238.178214,1094.502584
8,NaN,Micro-ATM (card+PIN),47.61107,176.598998,46.885785,165.610868,46.523256,169.789861,44.098577,165.978794,185.118688,677.978521,46.276707,175.274691,45.035104,160.147722,45.268623,166.896721,42.717585,163.325313,36.17688,130.669206,34.301233,114.196034,33.409083,114.69005,30.15,106.86,29.438437,105.485604,28.761351,98.190279,28.012455,99.08461,25.79,97.23,8.48,32.75,8.74,33.6,8.050844,29.67958
9,1.2,NFS - Cash deposit transactions,0.578889,5.856577,0.65626,6.204435,0.800842,7.527675,1.063142,9.313341,3.099133,28.902028,1.15338,10.128634,1.269453,10.793346,1.374606,12.089544,1.405829,12.697689,1.506552,13.148524,1.553448,13.633682,1.574175,14.628493,1.72,15.41,1.684215,15.886513,1.845935,14.727307,1.830103,12.724586,1.46,11.86,0.38,3.54,0.38,3.39,0.395898,3.305328



4. LAST 10 RAW ROWS


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41
40,15,eKYC Verification (Successful Txn),69.835093,NaN,86.967733,NaN,86.797157,NaN,96.844405,NaN,340.444388,NaN,100.704858,NaN,126.924549,NaN,118.577173,NaN,121.448672,NaN,107.327961,NaN,123.730323,NaN,112.241133,NaN,117.159865,0,113.724253,NaN,149.507458,0,157.744611,0,158.72505,NaN,50.777543,NaN,55.579376,NaN,59.84,NaN
41,16,Demographic Queries(Authenticated UID),12.746205,NaN,14.425978,NaN,14.29914,NaN,15.882859,NaN,57.354182,NaN,15.464665,NaN,21.628752,NaN,25.279737,NaN,26.75621,NaN,24.75597,NaN,32.86906,NaN,35.310288,NaN,37.780667,0,44.766081,NaN,55.756078,0,38.960434,0,42.801985,NaN,14.683099,NaN,16.05199,NaN,25.41,NaN
42,17,AEPS Tokenization,31.214191,NaN,13.76488,NaN,18.519702,NaN,13.446107,NaN,76.94488,NaN,12.06288,NaN,16.163595,NaN,20.381377,NaN,20.33227,NaN,17.334209,NaN,17.22768,NaN,23.420626,NaN,30.157681,0,38.273793,NaN,67.19309,0,159.011319,0,28.733299,NaN,7.045122,NaN,5.135637,NaN,7.53,NaN
43,18,BBPS (Bill Fetch),1367.39056,NaN,1555.775234,NaN,1630.905386,NaN,1547.534496,NaN,6101.605676,NaN,1735.525411,NaN,2008.040955,NaN,2596.6,NaN,2719.36,NaN,3166.211425,NaN,3751.18,NaN,4063.52,NaN,4076.57,0,4459.85,NaN,4661.77,0,4383.26,0,4327.103473,NaN,1567.15,NaN,1537.99,NaN,1567.378046,NaN
44,26,QSAM,0,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,0,NaN,0,NaN,0,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,NaN,Total Non Financial Txn (B),2570.044153,NaN,2747.331237,NaN,2843.598827,NaN,2781.632024,0,10942.606241,NaN,3025.066128,0,3482.589994,NaN,4073.789504,NaN,4203.443809,NaN,4635.385735,NaN,5334.684339,NaN,5669.559707,NaN,5693.503902,0,6031.727909,NaN,6404.674051,0,6133.195252,0,5920.994578,NaN,2068.52366,NaN,2048.234706,NaN,2129.237928,NaN
46,NaN,NaN,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,NaN,Total Financial + Non Financial (A+B),25559.87279,75382.917441,27898.625483,76343.637013,30882.282747,82907.074133,32807.864266,87733.386771,117148.645286,322367.015359,36251.455761,91869.44049,40469.473081,94436.483946,44713.784208,102491.806531,48252.077679,109778.171866,52105.626884,112929.045545,56038.363036,114604.770194,61008.499084,124295.397672,63707.147439,128028.549704,67485.370704,130812.391918,72242.795888,131382.095261,75521.889478,141847.86702,77112.013555,148125.533611,26477.348982,50637.102653,27287.223356,50747.182429,27012.54491,2920512.866412
48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,NaN,* NFS - ATM Cash Withdrawal - includes card+PIN transactions on micro-ATMs and does not include ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



5. RAW COLUMN INFORMATION


...


18. RAW DATA BASELINE


,Input_File,Sheet,Raw_Rows,Raw_Columns,Total_Cells,Total_Missing_Cells,Total_Present_Cells,Total_Numeric_Zero_Cells,Duplicate_Row_Instances,Completely_Empty_Rows,Completely_Empty_Columns,Numeric_Cells,String_Cells,Datetime_Cells,Boolean_Cells,Other_Cells
0,NPCI-Retail-Payments-Statistics---Excel-Format.xlsx,Sheet1,50,42,2100,554,1546,151,3,3,0,1380,160,0,0,6



STEP 1 VALIDATION
✓ Original dataset loaded successfully
✓ Raw dimensions recorded
✓ Missing values measured
✓ Zero values measured
✓ Duplicate rows checked
✓ Data types inspected
✓ Unique values inspected
✓ Baseline established
✓ NO DATA WAS MODIFIED
✓ NO VALUES WERE IMPUTED
✓ NO NaN VALUES WERE REPLACED
✓ NO ZERO VALUES WERE CHANGED

STEP 1 FINAL REPORT


,Input_File,Sheet,Raw_Rows,Raw_Columns,Total_Cells,Total_Missing_Cells,Total_Present_Cells,Total_Numeric_Zero_Cells,Duplicate_Row_Instances,Completely_Empty_Rows,Completely_Empty_Columns,Numeric_Cells,String_Cells,Datetime_Cells,Boolean_Cells,Other_Cells
0,NPCI-Retail-Payments-Statistics---Excel-Format.xlsx,Sheet1,50,42,2100,554,1546,151,3,3,0,1380,160,0,0,6



STEP 1 COMPLETE

Output report:
NPCI_Step1_Raw_Data_Audit_Report.xlsx

READY FOR STEP 2.


In [4]:
# ============================================================================
# STEP 2: HEADER RECONSTRUCTION & DATA STRUCTURE
# Dataset: NPCI Retail Payments Statistics
# ============================================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ----------------------------------------------------------------------------
# 1. INPUT FILE
# ----------------------------------------------------------------------------

INPUT_FILE = "NPCI-Retail-Payments-Statistics---Excel-Format.xlsx"
OUTPUT_FILE = "NPCI_Step2_Structured.xlsx"

print("=" * 90)
print("STEP 2: HEADER RECONSTRUCTION & DATA STRUCTURE")
print("=" * 90)

print("\nINPUT FILE")
print("-" * 90)
print(f"Input file : {INPUT_FILE}")

# ----------------------------------------------------------------------------
# 2. LOAD RAW EXCEL WITHOUT ASSUMING A HEADER
# ----------------------------------------------------------------------------

raw = pd.read_excel(
    INPUT_FILE,
    header=None
)

print("\nRAW DATA RELOADED")
print("-" * 90)
print(f"Rows    : {raw.shape[0]}")
print(f"Columns : {raw.shape[1]}")

assert raw.shape == (50, 42), (
    f"Unexpected raw dimensions: {raw.shape}. "
    f"Expected (50, 42)."
)

print("✓ Raw dimensions validated")

# ----------------------------------------------------------------------------
# 3. STRUCTURAL ROW IDENTIFICATION
# ----------------------------------------------------------------------------

TITLE_ROW = 0

FIN_HEADER_1 = 2
FIN_HEADER_2 = 3

FIN_DATA_START = 4
FIN_DATA_END = 32

NONFIN_SECTION_ROW = 33
NONFIN_HEADER_1 = 34
NONFIN_HEADER_2 = 35

NONFIN_DATA_START = 36
NONFIN_DATA_END = 46

FOOTNOTE_ROW = 48

print("\nSTRUCTURAL ROW IDENTIFICATION")
print("-" * 90)

print(f"Title row                    : {TITLE_ROW + 1}")
print(f"Financial header row 1       : {FIN_HEADER_1 + 1}")
print(f"Financial header row 2       : {FIN_HEADER_2 + 1}")
print(
    f"Financial data rows          : "
    f"{FIN_DATA_START + 1} - {FIN_DATA_END + 1}"
)
print(f"Non-financial section row    : {NONFIN_SECTION_ROW + 1}")
print(f"Non-financial header row 1   : {NONFIN_HEADER_1 + 1}")
print(f"Non-financial header row 2   : {NONFIN_HEADER_2 + 1}")
print(
    f"Non-financial data rows      : "
    f"{NONFIN_DATA_START + 1} - {NONFIN_DATA_END + 1}"
)
print(f"Footnote row                 : {FOOTNOTE_ROW + 1}")

# ----------------------------------------------------------------------------
# 4. HELPER FUNCTIONS
# ----------------------------------------------------------------------------

def is_blank(value):
    """
    True for NaN / None / empty strings.
    """
    if pd.isna(value):
        return True

    if isinstance(value, str) and value.strip() == "":
        return True

    return False


def normalize_period(value):
    """
    Convert the raw period/header value into a standardized period label.
    """

    if is_blank(value):
        return None

    # Datetime values such as 2026-04-01
    if isinstance(value, (pd.Timestamp,)):
        return value.strftime("%Y_%m")

    # Python datetime
    if hasattr(value, "strftime") and not isinstance(value, str):
        try:
            return value.strftime("%Y_%m")
        except Exception:
            pass

    text = str(value).strip()

    # Normalize common variants
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = text.replace(" ", "")

    # FY22-23 Q1
    m = re.match(
        r"FY(\d{2})-(\d{2})Q([1-4])$",
        text,
        flags=re.IGNORECASE
    )

    if m:
        y1, y2, q = m.groups()
        return f"FY_20{y1}_20{y2}_Q{q}"

    # F.Y-2022-23
    m = re.match(
        r"F\.?Y\.?-?(\d{4})-(\d{2})$",
        text,
        flags=re.IGNORECASE
    )

    if m:
        y1, y2 = m.groups()
        return f"FY_{y1}_20{y2}"

    # F.Y-2023-24 Q1
    m = re.match(
        r"F\.?Y\.?-?(\d{4})-(\d{2})Q([1-4])$",
        text,
        flags=re.IGNORECASE
    )

    if m:
        y1, y2, q = m.groups()
        return f"FY_{y1}_20{y2}_Q{q}"

    return text


def clean_header_text(value):
    """
    Clean Volume/Value header text.
    """

    if is_blank(value):
        return None

    text = str(value).strip()

    text = re.sub(r"\s+", " ", text)

    lower = text.lower()

    if "volume" in lower:
        return "Volume_in_Mn"

    if "value" in lower:
        return "Value_in_Bn"

    return None


# ----------------------------------------------------------------------------
# 5. READ THE TWO HEADER ROWS
# ----------------------------------------------------------------------------

header_row_1 = raw.iloc[FIN_HEADER_1]
header_row_2 = raw.iloc[FIN_HEADER_2]

print("\nPERIOD HEADER RECONSTRUCTION")
print("-" * 90)

# ----------------------------------------------------------------------------
# 6. FORWARD-FILL PERIOD HEADERS
# ----------------------------------------------------------------------------
#
# IMPORTANT:
# In the original Excel sheet, merged cells result in NaN in the following
# columns.
#
# Example:
#
# Q1 | Volume
#    | Value
#
# The Value column has no period in the raw sheet.
# Therefore, the Q1 period must be carried forward.
#
# ----------------------------------------------------------------------------

resolved_periods = []

current_period = None

for col_idx in range(raw.shape[1]):

    raw_period = header_row_1.iloc[col_idx]

    # If first header row contains a period, update current period.
    normalized = normalize_period(raw_period)

    if normalized is not None:
        current_period = normalized

    resolved_periods.append(current_period)

    sub_header = header_row_2.iloc[col_idx]

    print(
        f"Column {col_idx:02d} | "
        f"Period/Header = {repr(raw_period)} | "
        f"Sub-header = {repr(sub_header)} | "
        f"Resolved Period = {repr(current_period)}"
    )

# ----------------------------------------------------------------------------
# 7. SPECIAL CASE:
# THE UPLOADED SHEET MAY HAVE THE PERIOD INFORMATION IN HEADER ROW 2
# ----------------------------------------------------------------------------
#
# Your error output showed the period labels under "Sub-header".
# Therefore, if the first header row is entirely blank for transaction
# columns, detect the period from header row 2 and forward-fill from there.
#
# ----------------------------------------------------------------------------

transaction_start = 2

if all(
    is_blank(header_row_1.iloc[i])
    for i in range(transaction_start, raw.shape[1])
):
    print("\n⚠ Period labels detected in second header row.")
    print("✓ Reconstructing periods from the second header row.")

    resolved_periods = []
    current_period = None

    for col_idx in range(raw.shape[1]):

        candidate_1 = header_row_1.iloc[col_idx]
        candidate_2 = header_row_2.iloc[col_idx]

        period_candidate = normalize_period(candidate_1)

        if period_candidate is None:
            period_candidate = normalize_period(candidate_2)

        if period_candidate is not None:
            current_period = period_candidate

        resolved_periods.append(current_period)

# ----------------------------------------------------------------------------
# 8. DETERMINE VOLUME / VALUE
# ----------------------------------------------------------------------------

metric_labels = []

for col_idx in range(raw.shape[1]):

    h1 = clean_header_text(header_row_1.iloc[col_idx])
    h2 = clean_header_text(header_row_2.iloc[col_idx])

    metric = h1 if h1 is not None else h2

    metric_labels.append(metric)

# ----------------------------------------------------------------------------
# 9. HANDLE THE ACTUAL TWO-COLUMN PATTERN
# ----------------------------------------------------------------------------
#
# The dataset has:
#
# Period | Volume
#        | Value
#
# Therefore, if a transaction column does not explicitly contain Volume or
# Value, infer it from the alternating structure.
#
# Column 2 = Volume
# Column 3 = Value
# Column 4 = Volume
# Column 5 = Value
# ...
#
# This is structural reconstruction, NOT data modification.
# ----------------------------------------------------------------------------

for col_idx in range(2, raw.shape[1]):

    if metric_labels[col_idx] is None:

        if (col_idx - 2) % 2 == 0:
            metric_labels[col_idx] = "Volume_in_Mn"
        else:
            metric_labels[col_idx] = "Value_in_Bn"

# ----------------------------------------------------------------------------
# 10. VERIFY PERIODS AND METRICS
# ----------------------------------------------------------------------------

print("\nRECONSTRUCTED TRANSACTION HEADERS")
print("-" * 90)

for col_idx in range(2, raw.shape[1]):

    print(
        f"{col_idx:02d} | "
        f"Period = {resolved_periods[col_idx]} | "
        f"Metric = {metric_labels[col_idx]}"
    )

# ----------------------------------------------------------------------------
# 11. CREATE COLUMN NAMES
# ----------------------------------------------------------------------------

generated_columns = [
    "Sr_No",
    "NPCI_Operated_System"
]

for col_idx in range(2, raw.shape[1]):

    period = resolved_periods[col_idx]
    metric = metric_labels[col_idx]

    if period is None:
        period = f"Column_{col_idx:02d}_Unknown"

    if metric is None:
        metric = "Unknown"

    column_name = f"{period}_{metric}"

    generated_columns.append(column_name)

# ----------------------------------------------------------------------------
# 12. CLEAN COLUMN NAMES
# ----------------------------------------------------------------------------

def sanitize_column_name(name):

    name = str(name)

    name = name.replace(" ", "_")
    name = name.replace("-", "_")
    name = name.replace(".", "_")
    name = name.replace("/", "_")
    name = name.replace("(", "")
    name = name.replace(")", "")

    name = re.sub(r"_+", "_", name)

    return name.strip("_")


generated_columns = [
    sanitize_column_name(col)
    for col in generated_columns
]

# ----------------------------------------------------------------------------
# 13. DUPLICATE CHECK
# ----------------------------------------------------------------------------

duplicate_mask = pd.Series(generated_columns).duplicated(keep=False)

duplicate_columns = (
    pd.Series(generated_columns)[duplicate_mask]
    .unique()
    .tolist()
)

print("\nCOLUMN NAME VALIDATION")
print("-" * 90)

print(f"Total generated columns : {len(generated_columns)}")
print(f"Duplicate column names  : {len(duplicate_columns)}")

if duplicate_columns:

    print("\nDuplicate names detected:")
    for col in duplicate_columns:
        print(f"  {col}")

    raise ValueError(
        "Duplicate generated column names detected. "
        "Header reconstruction must be corrected."
    )

print("✓ Column names are unique")

# ----------------------------------------------------------------------------
# 14. PRINT GENERATED COLUMN NAMES
# ----------------------------------------------------------------------------

print("\nGENERATED COLUMN NAMES")
print("-" * 90)

for idx, col in enumerate(generated_columns):
    print(f"{idx:02d} | {col}")

# ----------------------------------------------------------------------------
# 15. EXTRACT FINANCIAL DATA
# ----------------------------------------------------------------------------

financial_data = raw.iloc[
    FIN_DATA_START:FIN_DATA_END + 1
].copy()

print("\nFINANCIAL SECTION")
print("-" * 90)

print(f"Rows    : {financial_data.shape[0]}")
print(f"Columns : {financial_data.shape[1]}")

# ----------------------------------------------------------------------------
# 16. EXTRACT NON-FINANCIAL DATA
# ----------------------------------------------------------------------------

nonfinancial_data = raw.iloc[
    NONFIN_DATA_START:NONFIN_DATA_END + 1
].copy()

print("\nNON-FINANCIAL SECTION")
print("-" * 90)

print(f"Rows    : {nonfinancial_data.shape[0]}")
print(f"Columns : {nonfinancial_data.shape[1]}")

# ----------------------------------------------------------------------------
# 17. COMBINE BOTH SECTIONS
# ----------------------------------------------------------------------------

structured = pd.concat(
    [
        financial_data,
        nonfinancial_data
    ],
    ignore_index=True
)

structured.columns = generated_columns

print("\nCOMBINED STRUCTURED DATA")
print("-" * 90)

print(f"Rows    : {structured.shape[0]}")
print(f"Columns : {structured.shape[1]}")

assert structured.shape == (40, 42)

# ----------------------------------------------------------------------------
# 18. COMPLETELY EMPTY ROW CHECK
# ----------------------------------------------------------------------------

empty_rows = structured.isna().all(axis=1).sum()

print("\nCOMPLETELY EMPTY ROW CHECK")
print("-" * 90)

print(f"Completely empty rows : {empty_rows}")

assert empty_rows == 0

print("✓ No completely empty structured rows")

# ----------------------------------------------------------------------------
# 19. TRANSACTION COLUMN IDENTIFICATION
# ----------------------------------------------------------------------------

transaction_columns = generated_columns[2:]

volume_columns = [
    col for col in transaction_columns
    if col.endswith("Volume_in_Mn")
]

value_columns = [
    col for col in transaction_columns
    if col.endswith("Value_in_Bn")
]

print("\nTRANSACTION COLUMN IDENTIFICATION")
print("-" * 90)

print(f"Transaction columns : {len(transaction_columns)}")
print(f"Volume columns      : {len(volume_columns)}")
print(f"Value columns       : {len(value_columns)}")

print("\nVolume columns:")

for col in volume_columns:
    print(f"  {col}")

print("\nValue columns:")

for col in value_columns:
    print(f"  {col}")

# ----------------------------------------------------------------------------
# 20. STRUCTURE VALIDATION
# ----------------------------------------------------------------------------

EXPECTED_TRANSACTION_COLUMNS = 40
EXPECTED_VOLUME_COLUMNS = 20
EXPECTED_VALUE_COLUMNS = 20

print("\nSTRUCTURE VALIDATION")
print("-" * 90)

print(
    f"Expected transaction columns : "
    f"{EXPECTED_TRANSACTION_COLUMNS}"
)

print(
    f"Actual transaction columns   : "
    f"{len(transaction_columns)}"
)

print(
    f"Expected volume columns      : "
    f"{EXPECTED_VOLUME_COLUMNS}"
)

print(
    f"Actual volume columns        : "
    f"{len(volume_columns)}"
)

print(
    f"Expected value columns       : "
    f"{EXPECTED_VALUE_COLUMNS}"
)

print(
    f"Actual value columns         : "
    f"{len(value_columns)}"
)

assert len(transaction_columns) == EXPECTED_TRANSACTION_COLUMNS
assert len(volume_columns) == EXPECTED_VOLUME_COLUMNS
assert len(value_columns) == EXPECTED_VALUE_COLUMNS

print("✓ Transaction column structure validated")

# ----------------------------------------------------------------------------
# 21. SAVE STRUCTURED DATA
# ----------------------------------------------------------------------------

structured.to_excel(
    OUTPUT_FILE,
    index=False
)

print("\nOUTPUT CHECKPOINT")
print("-" * 90)

print(f"Output file : {OUTPUT_FILE}")

# ----------------------------------------------------------------------------
# 22. FINAL REPORT
# ----------------------------------------------------------------------------

print("\n" + "=" * 90)
print("STEP 2 FINAL REPORT")
print("=" * 90)

print(
    f"""
Raw rows                         : {raw.shape[0]}
Raw columns                      : {raw.shape[1]}

Structured rows                 : {structured.shape[0]}
Structured columns              : {structured.shape[1]}

Transaction columns             : {len(transaction_columns)}
Volume columns                  : {len(volume_columns)}
Value columns                   : {len(value_columns)}

Completely empty rows           : {empty_rows}

Data values modified            : False
Missing values imputed          : False
NaN converted to zero           : False
Duplicate rows removed          : False
"""
)

print("=" * 90)
print("STEP 2 COMPLETE")
print("=" * 90)

print("✓ Header structure reconstructed")
print("✓ Merged period headers forward-filled")
print("✓ Financial section identified")
print("✓ Non-financial section identified")
print("✓ Period labels standardized")
print("✓ Volume columns identified")
print("✓ Value columns identified")
print("✓ Transaction columns identified")
print("✓ Column names validated")
print("✓ No data values modified")
print("✓ No missing values imputed")
print("✓ Existing zeros preserved")
print("✓ No rows deleted")
print("✓ No duplicate rows removed")
print("✓ Checkpoint saved")

print(f"\nOutput file: {OUTPUT_FILE}")
print("\nREADY FOR STEP 3.")

STEP 2: HEADER RECONSTRUCTION & DATA STRUCTURE

INPUT FILE
------------------------------------------------------------------------------------------
Input file : NPCI-Retail-Payments-Statistics---Excel-Format.xlsx

RAW DATA RELOADED
------------------------------------------------------------------------------------------
Rows    : 50
Columns : 42
✓ Raw dimensions validated

STRUCTURAL ROW IDENTIFICATION
------------------------------------------------------------------------------------------
Title row                    : 1
Financial header row 1       : 3
Financial header row 2       : 4
Financial data rows          : 5 - 33
Non-financial section row    : 34
Non-financial header row 1   : 35
Non-financial header row 2   : 36
Non-financial data rows      : 37 - 47
Footnote row                 : 49

PERIOD HEADER RECONSTRUCTION
------------------------------------------------------------------------------------------
Column 00 | Period/Header = nan | Sub-header = 'Sr. No.' | Resolved

In [12]:
# ============================================================
# STEP 3: DATA TYPE & FORMAT CLEANING
# ============================================================
#
# INPUT:
#     NPCI_Step2_Structured.xlsx
#
# OUTPUT:
#     NPCI_Step3_Cleaned.xlsx
#
# PURPOSE:
#     - Convert genuine transaction data to numeric dtype
#     - Preserve structural/header rows
#     - Preserve missing values
#     - Preserve existing zeros
#     - Preserve all transaction values
#     - Do NOT impute missing values
#     - Do NOT convert NaN to zero
#     - Do NOT delete rows
#     - Do NOT remove duplicates
#
# ============================================================

import pandas as pd
import numpy as np
import os


# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = "NPCI_Step2_Structured.xlsx"
OUTPUT_FILE = "NPCI_Step3_Cleaned.xlsx"


# ============================================================
# STEP 1 — LOAD INPUT FILE
# ============================================================

print("=" * 90)
print("STEP 3: DATA TYPE & FORMAT CLEANING")
print("=" * 90)

print("\nINPUT FILE")
print("-" * 90)
print(f"Input file : {INPUT_FILE}")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}"
    )

df = pd.read_excel(INPUT_FILE)

print(f"Rows       : {df.shape[0]}")
print(f"Columns    : {df.shape[1]}")


# ============================================================
# STEP 2 — VALIDATE INPUT DIMENSIONS
# ============================================================

assert df.shape == (40, 42), (
    f"Unexpected input dimensions: {df.shape}. "
    f"Expected (40, 42)."
)

print("\n✓ Input dimensions validated")


# ============================================================
# STEP 3 — MAKE DEEP COPY OF ORIGINAL DATA
# ============================================================

original_df = df.copy(deep=True)


# ============================================================
# STEP 4 — IDENTIFY TRANSACTION COLUMNS
# ============================================================

transaction_columns = [
    col
    for col in df.columns
    if "_Volume_in_Mn" in str(col)
    or "_Value_in_Bn" in str(col)
]

volume_columns = [
    col
    for col in transaction_columns
    if "_Volume_in_Mn" in str(col)
]

value_columns = [
    col
    for col in transaction_columns
    if "_Value_in_Bn" in str(col)
]


print("\nTRANSACTION COLUMN IDENTIFICATION")
print("-" * 90)

print(f"Transaction columns : {len(transaction_columns)}")
print(f"Volume columns      : {len(volume_columns)}")
print(f"Value columns       : {len(value_columns)}")


# ============================================================
# VALIDATE TRANSACTION STRUCTURE
# ============================================================

assert len(transaction_columns) == 40
assert len(volume_columns) == 20
assert len(value_columns) == 20

print("✓ Transaction column structure validated")


# ============================================================
# STEP 5 — IDENTIFY SYSTEM NAME COLUMN
# ============================================================

system_col = "NPCI_Operated_System"

assert system_col in df.columns

print("\nTEXT FORMAT CHECK")
print("-" * 90)

print(
    f"NPCI_Operated_System column present : "
    f"{system_col in df.columns}"
)


# ============================================================
# STEP 6 — IDENTIFY STRUCTURAL ROWS
# ============================================================
#
# There are two types of structural rows:
#
# 1. Rows with no system name
# 2. Rows containing embedded headers such as:
#       Volume (in Mn)
#       Value (in Bn)
#
# These rows MUST NOT be passed through numeric conversion.
#
# ============================================================

# ------------------------------------------------------------
# 6A. Rows without system name
# ------------------------------------------------------------

empty_system_mask = (
    df[system_col].isna()
    |
    df[system_col].astype(str).str.strip().eq("")
)

empty_system_rows = df.index[
    empty_system_mask
].tolist()


# ------------------------------------------------------------
# 6B. Rows containing embedded header labels
# ------------------------------------------------------------

header_labels = {
    "Volume (in Mn)",
    "Value (in Bn)",
    "Volume",
    "Value",
    "Volume (in Mn.)",
    "Value (in Bn.)"
}

header_row_mask = pd.Series(
    False,
    index=df.index
)

header_row_matches = {}

for idx in df.index:

    row_matches = []

    for col in transaction_columns:

        value = df.at[idx, col]

        if pd.isna(value):
            continue

        value_string = str(value).strip()

        if value_string in header_labels:

            header_row_mask.loc[idx] = True

            row_matches.append(
                (col, value_string)
            )

    if row_matches:
        header_row_matches[idx] = row_matches


header_label_rows = df.index[
    header_row_mask
].tolist()


# ------------------------------------------------------------
# 6C. Combine structural rows
# ------------------------------------------------------------

structural_rows = sorted(
    set(
        empty_system_rows
        +
        header_label_rows
    )
)


# ------------------------------------------------------------
# 6D. Genuine transaction rows
# ------------------------------------------------------------

transaction_rows = [
    idx
    for idx in df.index
    if idx not in structural_rows
]


print("\nSTRUCTURAL ROW IDENTIFICATION")
print("-" * 90)

print(
    f"Rows without system name : "
    f"{len(empty_system_rows)}"
)

print(
    f"Empty-system row indices : "
    f"{empty_system_rows}"
)

print(
    f"Embedded header rows     : "
    f"{len(header_label_rows)}"
)

print(
    f"Header row indices       : "
    f"{header_label_rows}"
)

print(
    f"Total structural rows    : "
    f"{len(structural_rows)}"
)

print(
    f"Structural row indices   : "
    f"{structural_rows}"
)

print(
    f"Transaction data rows    : "
    f"{len(transaction_rows)}"
)


# ============================================================
# STEP 7 — SHOW EMBEDDED HEADER ROWS
# ============================================================

if header_row_matches:

    print("\nEMBEDDED HEADER CONTENT DETECTED")
    print("-" * 90)

    for idx, matches in header_row_matches.items():

        print(
            f"Row {idx} : {matches}"
        )

else:

    print("\nEMBEDDED HEADER CONTENT DETECTED")
    print("-" * 90)

    print("None")


# ============================================================
# STEP 8 — CAPTURE ORIGINAL COUNTS
# ============================================================

# IMPORTANT:
# These counts are taken BEFORE any modification.

missing_before = (
    original_df[
        transaction_columns
    ]
    .isna()
    .sum()
    .sum()
)


zero_before = (
    original_df[
        transaction_columns
    ]
    .eq(0)
    .sum()
    .sum()
)


print("\nORIGINAL DATA STATE")
print("-" * 90)

print(
    f"Missing transaction cells : "
    f"{missing_before}"
)

print(
    f"Zero transaction cells    : "
    f"{zero_before}"
)


# ============================================================
# STEP 9 — SAVE ORIGINAL TRANSACTION DATA
# ============================================================
#
# Only genuine transaction rows are compared for exact
# value preservation.
#
# Structural rows are separately checked.
#
# ============================================================

original_transaction_data = original_df.loc[
    transaction_rows,
    transaction_columns
].copy(deep=True)


original_structural_data = original_df.loc[
    structural_rows,
    transaction_columns
].copy(deep=True)


# ============================================================
# STEP 10 — CHECK NON-NUMERIC VALUES BEFORE CONVERSION
# ============================================================
#
# A value is considered invalid only when:
#
#     - it is in a genuine transaction row
#     - it is not missing
#     - it cannot be interpreted as a number
#
# Structural/header rows are excluded.
#
# ============================================================

non_numeric_before = {}

for col in transaction_columns:

    series = df.loc[
        transaction_rows,
        col
    ]

    numeric_test = pd.to_numeric(
        series,
        errors="coerce"
    )

    invalid_mask = (
        series.notna()
        &
        numeric_test.isna()
    )

    count_invalid = int(
        invalid_mask.sum()
    )

    if count_invalid > 0:

        non_numeric_before[col] = count_invalid


print("\nNON-NUMERIC TRANSACTION VALUE CHECK")
print("-" * 90)

if non_numeric_before:

    print(
        "⚠ Non-numeric values detected in genuine "
        "transaction rows:"
    )

    for col, count in non_numeric_before.items():

        print(
            f"  {col} : {count}"
        )

else:

    print(
        "✓ No non-numeric values detected in "
        "genuine transaction rows."
    )


# ============================================================
# STEP 11 — DISPLAY EXACT INVALID VALUES IF ANY
# ============================================================

if non_numeric_before:

    print("\nINVALID VALUE DETAILS")
    print("-" * 90)

    for col in non_numeric_before:

        series = df.loc[
            transaction_rows,
            col
        ]

        numeric_test = pd.to_numeric(
            series,
            errors="coerce"
        )

        invalid_mask = (
            series.notna()
            &
            numeric_test.isna()
        )

        invalid_rows = series[
            invalid_mask
        ]

        for idx, value in invalid_rows.items():

            print(
                f"Row {idx} | "
                f"Column {col} | "
                f"Value = {repr(value)}"
            )


# ============================================================
# STEP 12 — STOP IF REAL INVALID DATA EXISTS
# ============================================================
#
# We do NOT use errors="coerce" for conversion because that
# would silently turn bad strings into NaN.
#
# ============================================================

if non_numeric_before:

    raise ValueError(
        "\nReal non-numeric transaction values exist "
        "inside genuine transaction rows.\n"
        "The data has NOT been modified.\n"
        "Inspect the INVALID VALUE DETAILS above."
    )


# ============================================================
# STEP 13 — NUMERIC DATA TYPE CONVERSION
# ============================================================

print("\nNUMERIC DATA TYPE CONVERSION")
print("-" * 90)

for col in transaction_columns:

    # --------------------------------------------------------
    # Convert ONLY genuine transaction rows.
    #
    # Structural rows are completely untouched.
    #
    # errors='raise' guarantees that unexpected bad data
    # cannot silently become NaN.
    # --------------------------------------------------------

    converted = pd.to_numeric(
        df.loc[
            transaction_rows,
            col
        ],
        errors="raise"
    )

    df.loc[
        transaction_rows,
        col
    ] = converted


print("✓ Numeric transaction data converted safely.")
print("✓ Structural rows preserved.")
print("✓ Structural row transaction cells preserved.")


# ============================================================
# STEP 14 — VALIDATE STRUCTURAL ROWS
# ============================================================

structural_rows_match = original_structural_data.equals(
    df.loc[
        structural_rows,
        transaction_columns
    ]
)


assert structural_rows_match

print(
    "✓ Structural row values unchanged."
)


# ============================================================
# STEP 15 — VALIDATE TRANSACTION DATA TYPES
# ============================================================

numeric_transaction_cells = 0
non_numeric_transaction_cells = 0

for col in transaction_columns:

    series = df.loc[
        transaction_rows,
        col
    ]

    numeric_mask = pd.to_numeric(
        series,
        errors="coerce"
    ).notna()

    missing_mask = series.isna()

    valid_mask = (
        numeric_mask
        |
        missing_mask
    )

    non_numeric_transaction_cells += int(
        (~valid_mask).sum()
    )

    numeric_transaction_cells += int(
        numeric_mask.sum()
    )


print("\nTRANSACTION DATA TYPE VALIDATION")
print("-" * 90)

print(
    f"Transaction columns checked       : "
    f"{len(transaction_columns)}"
)

print(
    f"Numeric transaction data cells    : "
    f"{numeric_transaction_cells}"
)

print(
    f"Non-numeric transaction cells     : "
    f"{non_numeric_transaction_cells}"
)


assert non_numeric_transaction_cells == 0

print(
    "✓ All actual transaction data cells "
    "are numeric or missing."
)


# ============================================================
# STEP 16 — SR_NO VALIDATION
# ============================================================

print("\nSr_No CHECK")
print("-" * 90)

print(
    f"Sr_No dtype : "
    f"{df['Sr_No'].dtype}"
)

# We intentionally DO NOT convert Sr_No.
assert (
    df["Sr_No"].dtype
    ==
    original_df["Sr_No"].dtype
)

print(
    "✓ Sr_No structure preserved."
)


# ============================================================
# STEP 17 — NEGATIVE VALUE CHECK
# ============================================================

negative_volume_cells = (
    df.loc[
        transaction_rows,
        volume_columns
    ]
    .lt(0)
    .sum()
    .sum()
)

negative_value_cells = (
    df.loc[
        transaction_rows,
        value_columns
    ]
    .lt(0)
    .sum()
    .sum()
)


print("\nNEGATIVE VALUE CHECK")
print("-" * 90)

print(
    f"Negative volume cells : "
    f"{negative_volume_cells}"
)

print(
    f"Negative value cells  : "
    f"{negative_value_cells}"
)


assert negative_volume_cells == 0
assert negative_value_cells == 0

print(
    "✓ No negative transaction values detected."
)


# ============================================================
# STEP 18 — INFINITY CHECK
# ============================================================

positive_infinity_cells = 0
negative_infinity_cells = 0

for col in transaction_columns:

    series = df.loc[
        transaction_rows,
        col
    ]

    numeric_series = pd.to_numeric(
        series,
        errors="coerce"
    )

    positive_infinity_cells += int(
        np.isposinf(
            numeric_series
        ).sum()
    )

    negative_infinity_cells += int(
        np.isneginf(
            numeric_series
        ).sum()
    )


print("\nINFINITY CHECK")
print("-" * 90)

print(
    f"Positive infinity cells : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells : "
    f"{negative_infinity_cells}"
)


assert positive_infinity_cells == 0
assert negative_infinity_cells == 0


# ============================================================
# STEP 19 — MISSINGNESS VALIDATION
# ============================================================

missing_after = (
    df[
        transaction_columns
    ]
    .isna()
    .sum()
    .sum()
)


missingness_preserved = (
    missing_before
    ==
    missing_after
)


# ============================================================
# STEP 20 — ZERO VALIDATION
# ============================================================

zero_after = (
    df[
        transaction_columns
    ]
    .eq(0)
    .sum()
    .sum()
)


zero_count_preserved = (
    zero_before
    ==
    zero_after
)


# ============================================================
# STEP 21 — EXACT TRANSACTION VALUE COMPARISON
# ============================================================
#
# Numeric conversion should NOT change the actual values.
#
# NaN == NaN normally evaluates False, so pandas .equals()
# is used because it treats corresponding missing values
# correctly.
#
# ============================================================

after_transaction_data = df.loc[
    transaction_rows,
    transaction_columns
].copy(deep=True)


transaction_values_match = (
    original_transaction_data.equals(
        after_transaction_data
    )
)


# ============================================================
# STEP 22 — DATA MODIFICATION FLAG
# ============================================================

data_values_modified = (
    not transaction_values_match
)


# ============================================================
# STEP 23 — IMPUTATION FLAG
# ============================================================
#
# Step 3 NEVER imputes.
#
# ============================================================

imputation_performed = False

nan_converted_to_zero = False


# ============================================================
# STEP 24 — DATA TYPE REPORT
# ============================================================

before_dtypes = original_df.dtypes
after_dtypes = df.dtypes

dtype_report = pd.DataFrame({
    "Column": df.columns,
    "Before_Data_Type": [
        str(before_dtypes[col])
        for col in df.columns
    ],
    "After_Data_Type": [
        str(after_dtypes[col])
        for col in df.columns
    ]
})


print("\nDATA TYPE REPORT")
print("-" * 90)

print(
    dtype_report.to_string(
        index=False
    )
)


# ============================================================
# STEP 25 — FINAL VALIDATION
# ============================================================

print("\n" + "=" * 90)
print("STEP 3 VALIDATION")
print("=" * 90)

print(
    f"Missing cells before       : "
    f"{missing_before}"
)

print(
    f"Missing cells after        : "
    f"{missing_after}"
)

print(
    f"Missingness preserved      : "
    f"{missingness_preserved}"
)

print()

print(
    f"Zero cells before          : "
    f"{zero_before}"
)

print(
    f"Zero cells after           : "
    f"{zero_after}"
)

print(
    f"Zero count preserved      : "
    f"{zero_count_preserved}"
)

print()

if transaction_values_match:

    print(
        "Transaction value mismatches : 0"
    )

else:

    print(
        "Transaction value mismatches : DETECTED"
    )

print(
    f"Data values modified         : "
    f"{data_values_modified}"
)

print(
    f"Imputation performed         : "
    f"{imputation_performed}"
)

print(
    f"NaN converted to zero        : "
    f"{nan_converted_to_zero}"
)


# ============================================================
# STEP 26 — HARD ASSERTIONS
# ============================================================

# Structure
assert len(transaction_columns) == 40
assert len(volume_columns) == 20
assert len(value_columns) == 20

# Actual transaction data
assert non_numeric_transaction_cells == 0

# Missingness
assert missingness_preserved

# Existing zeros
assert zero_count_preserved

# Exact values
assert transaction_values_match

# No imputation
assert imputation_performed is False

# No NaN → zero conversion
assert nan_converted_to_zero is False

# No negatives
assert negative_volume_cells == 0
assert negative_value_cells == 0

# No infinity
assert positive_infinity_cells == 0
assert negative_infinity_cells == 0

# Structural rows
assert structural_rows_match


# ============================================================
# STEP 27 — SAVE OUTPUT
# ============================================================

df.to_excel(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# STEP 28 — VERIFY OUTPUT FILE
# ============================================================

assert os.path.exists(
    OUTPUT_FILE
)


# Reload saved output and verify dimensions
saved_df = pd.read_excel(
    OUTPUT_FILE
)

assert saved_df.shape == (
    40,
    42
)


# ============================================================
# STEP 29 — FINAL REPORT
# ============================================================

print("\n" + "=" * 90)
print("STEP 3 FINAL REPORT")
print("=" * 90)

print(
    f"Input file                      : "
    f"{INPUT_FILE}"
)

print(
    f"Output file                     : "
    f"{OUTPUT_FILE}"
)

print()

print(
    f"Rows                            : "
    f"{df.shape[0]}"
)

print(
    f"Columns                         : "
    f"{df.shape[1]}"
)

print()

print(
    f"Transaction columns             : "
    f"{len(transaction_columns)}"
)

print(
    f"Volume columns                  : "
    f"{len(volume_columns)}"
)

print(
    f"Value columns                   : "
    f"{len(value_columns)}"
)

print()

print(
    f"Structural rows                 : "
    f"{len(structural_rows)}"
)

print(
    f"Transaction data rows           : "
    f"{len(transaction_rows)}"
)

print()

print(
    f"Missing cells before            : "
    f"{missing_before}"
)

print(
    f"Missing cells after             : "
    f"{missing_after}"
)

print(
    f"Missingness preserved            : "
    f"{missingness_preserved}"
)

print()

print(
    f"Zero cells before               : "
    f"{zero_before}"
)

print(
    f"Zero cells after                : "
    f"{zero_after}"
)

print(
    f"Zero count preserved            : "
    f"{zero_count_preserved}"
)

print()

print(
    f"Numeric transaction cells       : "
    f"{numeric_transaction_cells}"
)

print(
    f"Non-numeric transaction cells   : "
    f"{non_numeric_transaction_cells}"
)

print()

print(
    f"Transaction values unchanged    : "
    f"{transaction_values_match}"
)

print(
    f"Data values modified            : "
    f"{data_values_modified}"
)

print(
    f"Imputation performed            : "
    f"{imputation_performed}"
)

print(
    f"NaN converted to zero           : "
    f"{nan_converted_to_zero}"
)

print()

print(
    f"Negative volume cells           : "
    f"{negative_volume_cells}"
)

print(
    f"Negative value cells            : "
    f"{negative_value_cells}"
)

print(
    f"Positive infinity cells        : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells        : "
    f"{negative_infinity_cells}"
)

print()

print("=" * 90)
print("STEP 3 COMPLETE")
print("=" * 90)

print("✓ Input dimensions validated")
print("✓ Transaction columns identified")
print("✓ Structural rows identified")
print("✓ Embedded header rows identified")
print("✓ Structural rows preserved")
print("✓ Numeric transaction data converted")
print("✓ Missing values preserved")
print("✓ Existing zeros preserved")
print("✓ No values imputed")
print("✓ No NaN converted to zero")
print("✓ No negative transaction values")
print("✓ No infinity values")
print("✓ Transaction values unchanged")
print("✓ Validation passed")
print("✓ Checkpoint saved")

print()
print(
    f"Output file: {OUTPUT_FILE}"
)

print("=" * 90)

STEP 3: DATA TYPE & FORMAT CLEANING

INPUT FILE
------------------------------------------------------------------------------------------
Input file : NPCI_Step2_Structured.xlsx
Rows       : 40
Columns    : 42

✓ Input dimensions validated

TRANSACTION COLUMN IDENTIFICATION
------------------------------------------------------------------------------------------
Transaction columns : 40
Volume columns      : 20
Value columns       : 20
✓ Transaction column structure validated

TEXT FORMAT CHECK
------------------------------------------------------------------------------------------
NPCI_Operated_System column present : True

STRUCTURAL ROW IDENTIFICATION
------------------------------------------------------------------------------------------
Rows without system name : 2
Empty-system row indices : [29, 39]
Embedded header rows     : 2
Header row indices       : [0, 29]
Total structural rows    : 3
Structural row indices   : [0, 29, 39]
Transaction data rows    : 37

EMBEDDED HEADE

In [13]:
# =============================================================================
# STEP 4: MISSING VALUE ANALYSIS & CONTROLLED CLEANING
# =============================================================================
#
# INPUT :
#     NPCI_Step3_Cleaned.xlsx
#
# OUTPUT:
#     NPCI_Step4_MissingValue_Handled.xlsx
#
# PRINCIPLES:
#     1. Preserve all structural / embedded-header rows.
#     2. Never convert missing values to zero automatically.
#     3. Never modify existing transaction values.
#     4. Never modify existing zeros.
#     5. Identify missing values separately from genuine zeros.
#     6. Do not perform statistical imputation at this stage.
#     7. Preserve the original row/column structure.
#
# =============================================================================


import pandas as pd
import numpy as np
import os
import re
from datetime import datetime


# =============================================================================
# CONFIGURATION
# =============================================================================

INPUT_FILE = "NPCI_Step3_Cleaned.xlsx"
OUTPUT_FILE = "NPCI_Step4_MissingValue_Handled.xlsx"


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("STEP 4: MISSING VALUE ANALYSIS & CONTROLLED CLEANING")
print("=" * 90)


# =============================================================================
# INPUT VALIDATION
# =============================================================================

print("\nINPUT FILE")
print("-" * 90)
print(f"Input file : {INPUT_FILE}")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}"
    )

df = pd.read_excel(INPUT_FILE)

print(f"Rows       : {df.shape[0]}")
print(f"Columns    : {df.shape[1]}")


# =============================================================================
# EXPECTED DIMENSIONS
# =============================================================================

assert df.shape == (40, 42), (
    f"Unexpected input dimensions: {df.shape}. "
    f"Expected (40, 42)."
)

print("✓ Input dimensions validated")


# =============================================================================
# EXPECTED CORE COLUMNS
# =============================================================================

required_columns = [
    "Sr_No",
    "NPCI_Operated_System"
]

for col in required_columns:
    assert col in df.columns, f"Required column missing: {col}"

print("\nCORE COLUMN VALIDATION")
print("-" * 90)
print("✓ Sr_No present")
print("✓ NPCI_Operated_System present")


# =============================================================================
# TRANSACTION COLUMN IDENTIFICATION
# =============================================================================

transaction_columns = [
    col for col in df.columns
    if (
        "_Volume_in_Mn" in str(col)
        or "_Value_in_Bn" in str(col)
    )
]

volume_columns = [
    col for col in transaction_columns
    if "_Volume_in_Mn" in str(col)
]

value_columns = [
    col for col in transaction_columns
    if "_Value_in_Bn" in str(col)
]


print("\nTRANSACTION COLUMN IDENTIFICATION")
print("-" * 90)
print(f"Transaction columns : {len(transaction_columns)}")
print(f"Volume columns      : {len(volume_columns)}")
print(f"Value columns       : {len(value_columns)}")


assert len(transaction_columns) == 40
assert len(volume_columns) == 20
assert len(value_columns) == 20

print("✓ Transaction column structure validated")


# =============================================================================
# CREATE ORIGINAL COPY
# =============================================================================
#
# This copy is used for all before/after integrity checks.
#

df_original = df.copy(deep=True)


# =============================================================================
# STRUCTURAL ROW IDENTIFICATION
# =============================================================================
#
# There are two kinds of structural rows:
#
#   A. Embedded header rows:
#      contain "Volume (in Mn)" / "Value (in Bn)"
#
#   B. Rows without a system name:
#      blank NPCI_Operated_System
#
# These rows must NEVER be treated as transaction records.
# =============================================================================


system_series = df["NPCI_Operated_System"]


# Detect embedded headers
embedded_header_rows = []

for idx in df.index:

    row_values = df.loc[idx, transaction_columns]

    row_text = " ".join(
        str(x).strip()
        for x in row_values
        if pd.notna(x)
    ).lower()

    if (
        "volume (in mn)" in row_text
        or "value (in bn)" in row_text
    ):
        embedded_header_rows.append(idx)


# Detect rows without system name
empty_system_rows = []

for idx in df.index:

    value = df.loc[idx, "NPCI_Operated_System"]

    if pd.isna(value) or str(value).strip() == "":
        empty_system_rows.append(idx)


# Combined structural rows
structural_rows = sorted(
    set(embedded_header_rows + empty_system_rows)
)

transaction_rows = [
    idx for idx in df.index
    if idx not in structural_rows
]


print("\nSTRUCTURAL ROW IDENTIFICATION")
print("-" * 90)
print(f"Embedded header rows : {len(embedded_header_rows)}")
print(f"Embedded headers    : {embedded_header_rows}")

print(f"Empty-system rows    : {len(empty_system_rows)}")
print(f"Empty-system indices : {empty_system_rows}")

print(f"Structural rows      : {len(structural_rows)}")
print(f"Structural indices   : {structural_rows}")

print(f"Transaction rows     : {len(transaction_rows)}")

assert len(embedded_header_rows) == 2
assert len(empty_system_rows) == 2
assert len(structural_rows) == 3
assert len(transaction_rows) == 37

print("✓ Structural rows identified correctly")


# =============================================================================
# ORIGINAL MISSINGNESS ANALYSIS
# =============================================================================

print("\nORIGINAL MISSING VALUE ANALYSIS")
print("-" * 90)


# Missing transaction cells ONLY in genuine transaction rows
missing_transaction_before = (
    df_original.loc[
        transaction_rows,
        transaction_columns
    ].isna().sum().sum()
)


# Existing zero transaction cells
zero_transaction_before = (
    (
        df_original.loc[
            transaction_rows,
            transaction_columns
        ] == 0
    )
    .sum()
    .sum()
)


# Total missing cells in complete dataframe
missing_total_before = int(
    df_original.isna().sum().sum()
)


print(
    f"Missing transaction cells : "
    f"{missing_transaction_before}"
)

print(
    f"Zero transaction cells    : "
    f"{zero_transaction_before}"
)

print(
    f"Total missing cells       : "
    f"{missing_total_before}"
)


# =============================================================================
# MISSING VALUES BY COLUMN
# =============================================================================

missing_by_column_before = (
    df_original.loc[
        transaction_rows,
        transaction_columns
    ]
    .isna()
    .sum()
)


print("\nMISSING VALUES BY TRANSACTION COLUMN")
print("-" * 90)

for col, count in missing_by_column_before.items():

    if count > 0:
        print(f"{col:<55} : {int(count)}")


# =============================================================================
# MISSING VALUES BY TRANSACTION ROW
# =============================================================================

missing_by_row_before = (
    df_original.loc[
        transaction_rows,
        transaction_columns
    ]
    .isna()
    .sum(axis=1)
)


print("\nMISSING VALUES BY TRANSACTION ROW")
print("-" * 90)

rows_with_missing = (
    missing_by_row_before[
        missing_by_row_before > 0
    ]
)

print(
    f"Transaction rows containing missing values : "
    f"{len(rows_with_missing)}"
)


# =============================================================================
# MISSING VALUE PATTERN ANALYSIS
# =============================================================================

print("\nMISSING VALUE PATTERN ANALYSIS")
print("-" * 90)


# Count rows with all transaction values missing
all_missing_rows = []

# Count rows with partially missing transaction values
partial_missing_rows = []

# Count complete rows
complete_transaction_rows = []

for idx in transaction_rows:

    row = df_original.loc[idx, transaction_columns]

    missing_count = row.isna().sum()

    if missing_count == len(transaction_columns):

        all_missing_rows.append(idx)

    elif missing_count > 0:

        partial_missing_rows.append(idx)

    else:

        complete_transaction_rows.append(idx)


print(
    f"Rows with all transaction values missing : "
    f"{len(all_missing_rows)}"
)

print(
    f"Rows with partial missing values          : "
    f"{len(partial_missing_rows)}"
)

print(
    f"Rows with complete transaction data       : "
    f"{len(complete_transaction_rows)}"
)


# =============================================================================
# DECISION
# =============================================================================
#
# IMPORTANT:
#
# No automatic imputation is performed.
#
# Reasons:
#
#   - A missing transaction value is not necessarily zero.
#   - Forward fill could incorrectly copy values between payment systems.
#   - Mean/median imputation would create artificial financial data.
#   - Zero replacement would change the meaning of the source data.
#
# Therefore Step 4 records and validates missingness but preserves it.
# =============================================================================


print("\nMISSING VALUE HANDLING DECISION")
print("-" * 90)

print("✓ Missing values identified.")
print("✓ Missing values are NOT imputed.")
print("✓ Missing values are NOT converted to zero.")
print("✓ Existing zeros are preserved.")
print("✓ Structural rows are preserved.")
print("✓ Original transaction values are preserved.")


# =============================================================================
# NO DATA MODIFICATION
# =============================================================================
#
# Make a deep copy and intentionally leave missing values unchanged.
#

df_cleaned = df_original.copy(deep=True)


# =============================================================================
# DATA TYPE VALIDATION
# =============================================================================

print("\nTRANSACTION DATA VALIDATION")
print("-" * 90)


non_numeric_cells = 0
numeric_cells = 0

for col in transaction_columns:

    values = df_cleaned.loc[
        transaction_rows,
        col
    ]

    numeric_values = pd.to_numeric(
        values,
        errors="coerce"
    )

    # Values that are neither numeric nor missing
    invalid_mask = (
        values.notna()
        & numeric_values.isna()
    )

    invalid_count = int(invalid_mask.sum())

    non_numeric_cells += invalid_count

    numeric_cells += int(
        numeric_values.notna().sum()
    )


print(
    f"Numeric transaction cells    : "
    f"{numeric_cells}"
)

print(
    f"Non-numeric transaction cells : "
    f"{non_numeric_cells}"
)

assert non_numeric_cells == 0

print(
    "✓ All genuine transaction cells are "
    "numeric or missing."
)


# =============================================================================
# NEGATIVE VALUE CHECK
# =============================================================================

print("\nNEGATIVE VALUE CHECK")
print("-" * 90)


negative_volume_cells = 0
negative_value_cells = 0


for col in volume_columns:

    values = pd.to_numeric(
        df_cleaned.loc[
            transaction_rows,
            col
        ],
        errors="coerce"
    )

    negative_volume_cells += int(
        (values < 0).sum()
    )


for col in value_columns:

    values = pd.to_numeric(
        df_cleaned.loc[
            transaction_rows,
            col
        ],
        errors="coerce"
    )

    negative_value_cells += int(
        (values < 0).sum()
    )


print(
    f"Negative volume cells : "
    f"{negative_volume_cells}"
)

print(
    f"Negative value cells  : "
    f"{negative_value_cells}"
)

assert negative_volume_cells == 0
assert negative_value_cells == 0

print("✓ No negative transaction values detected.")


# =============================================================================
# INFINITY CHECK
# =============================================================================

print("\nINFINITY CHECK")
print("-" * 90)


positive_infinity_cells = 0
negative_infinity_cells = 0


for col in transaction_columns:

    values = pd.to_numeric(
        df_cleaned.loc[
            transaction_rows,
            col
        ],
        errors="coerce"
    )

    positive_infinity_cells += int(
        np.isposinf(values).sum()
    )

    negative_infinity_cells += int(
        np.isneginf(values).sum()
    )


print(
    f"Positive infinity cells : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells : "
    f"{negative_infinity_cells}"
)

assert positive_infinity_cells == 0
assert negative_infinity_cells == 0

print("✓ No infinity values detected.")


# =============================================================================
# STRUCTURAL ROW INTEGRITY CHECK
# =============================================================================

print("\nSTRUCTURAL ROW INTEGRITY CHECK")
print("-" * 90)


structural_values_match = True

for idx in structural_rows:

    original_row = df_original.loc[
        idx,
        transaction_columns
    ]

    cleaned_row = df_cleaned.loc[
        idx,
        transaction_columns
    ]

    if not original_row.equals(cleaned_row):
        structural_values_match = False
        print(
            f"⚠ Structural row changed: {idx}"
        )


assert structural_values_match

print("✓ Structural transaction cells unchanged.")


# =============================================================================
# TRANSACTION VALUE INTEGRITY CHECK
# =============================================================================

print("\nTRANSACTION VALUE INTEGRITY CHECK")
print("-" * 90)


transaction_value_mismatches = 0


for idx in transaction_rows:

    for col in transaction_columns:

        original_value = df_original.loc[idx, col]
        cleaned_value = df_cleaned.loc[idx, col]

        # Both missing
        if pd.isna(original_value) and pd.isna(cleaned_value):
            continue

        # One missing and one non-missing
        if pd.isna(original_value) != pd.isna(cleaned_value):
            transaction_value_mismatches += 1
            continue

        # Numeric comparison
        try:

            original_numeric = float(original_value)
            cleaned_numeric = float(cleaned_value)

            if not np.isclose(
                original_numeric,
                cleaned_numeric,
                equal_nan=True
            ):
                transaction_value_mismatches += 1

        except (ValueError, TypeError):

            if original_value != cleaned_value:
                transaction_value_mismatches += 1


print(
    f"Transaction value mismatches : "
    f"{transaction_value_mismatches}"
)

assert transaction_value_mismatches == 0

print("✓ All transaction values unchanged.")


# =============================================================================
# MISSINGNESS AFTER CLEANING
# =============================================================================

missing_transaction_after = (
    df_cleaned.loc[
        transaction_rows,
        transaction_columns
    ].isna().sum().sum()
)


zero_transaction_after = (
    (
        df_cleaned.loc[
            transaction_rows,
            transaction_columns
        ] == 0
    )
    .sum()
    .sum()
)


missing_total_after = int(
    df_cleaned.isna().sum().sum()
)


# =============================================================================
# VALIDATION FLAGS
# =============================================================================

missingness_preserved = (
    missing_transaction_before
    == missing_transaction_after
)

zero_count_preserved = (
    zero_transaction_before
    == zero_transaction_after
)

transaction_values_match = (
    transaction_value_mismatches == 0
)

row_count_preserved = (
    df_original.shape[0]
    == df_cleaned.shape[0]
)

column_count_preserved = (
    df_original.shape[1]
    == df_cleaned.shape[1]
)


# =============================================================================
# FINAL VALIDATION
# =============================================================================

print("\n" + "=" * 90)
print("STEP 4 VALIDATION")
print("=" * 90)

print(
    f"Missing transaction cells before : "
    f"{missing_transaction_before}"
)

print(
    f"Missing transaction cells after  : "
    f"{missing_transaction_after}"
)

print(
    f"Missingness preserved             : "
    f"{missingness_preserved}"
)

print()

print(
    f"Zero transaction cells before     : "
    f"{zero_transaction_before}"
)

print(
    f"Zero transaction cells after      : "
    f"{zero_transaction_after}"
)

print(
    f"Zero count preserved              : "
    f"{zero_count_preserved}"
)

print()

print(
    f"Transaction value mismatches      : "
    f"{transaction_value_mismatches}"
)

print(
    f"Transaction values unchanged      : "
    f"{transaction_values_match}"
)

print(
    f"Rows preserved                    : "
    f"{row_count_preserved}"
)

print(
    f"Columns preserved                 : "
    f"{column_count_preserved}"
)


# =============================================================================
# ASSERTIONS
# =============================================================================

assert row_count_preserved
assert column_count_preserved

assert missingness_preserved
assert zero_count_preserved

assert transaction_values_match

assert structural_values_match

assert non_numeric_cells == 0

assert negative_volume_cells == 0
assert negative_value_cells == 0

assert positive_infinity_cells == 0
assert negative_infinity_cells == 0


# =============================================================================
# SAVE CHECKPOINT
# =============================================================================

df_cleaned.to_excel(
    OUTPUT_FILE,
    index=False
)


# =============================================================================
# VERIFY SAVED FILE
# =============================================================================

if not os.path.exists(OUTPUT_FILE):
    raise IOError(
        f"Output file was not created: {OUTPUT_FILE}"
    )


# Reload output for final checkpoint verification
df_saved = pd.read_excel(OUTPUT_FILE)


assert df_saved.shape == df_original.shape


# Verify transaction values after Excel round-trip
saved_transaction_mismatches = 0

for idx in transaction_rows:

    for col in transaction_columns:

        original_value = df_original.loc[idx, col]
        saved_value = df_saved.loc[idx, col]

        if pd.isna(original_value) and pd.isna(saved_value):
            continue

        if pd.isna(original_value) != pd.isna(saved_value):
            saved_transaction_mismatches += 1
            continue

        try:

            if not np.isclose(
                float(original_value),
                float(saved_value),
                equal_nan=True
            ):
                saved_transaction_mismatches += 1

        except (ValueError, TypeError):

            if original_value != saved_value:
                saved_transaction_mismatches += 1


assert saved_transaction_mismatches == 0


# =============================================================================
# FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("STEP 4 FINAL REPORT")
print("=" * 90)

print(
    f"Input file                       : "
    f"{INPUT_FILE}"
)

print(
    f"Output file                      : "
    f"{OUTPUT_FILE}"
)

print()

print(
    f"Rows                             : "
    f"{df_cleaned.shape[0]}"
)

print(
    f"Columns                          : "
    f"{df_cleaned.shape[1]}"
)

print()

print(
    f"Transaction columns              : "
    f"{len(transaction_columns)}"
)

print(
    f"Volume columns                   : "
    f"{len(volume_columns)}"
)

print(
    f"Value columns                    : "
    f"{len(value_columns)}"
)

print()

print(
    f"Structural rows                  : "
    f"{len(structural_rows)}"
)

print(
    f"Transaction data rows            : "
    f"{len(transaction_rows)}"
)

print()

print(
    f"Missing transaction cells before : "
    f"{missing_transaction_before}"
)

print(
    f"Missing transaction cells after  : "
    f"{missing_transaction_after}"
)

print(
    f"Missingness preserved             : "
    f"{missingness_preserved}"
)

print()

print(
    f"Zero transaction cells before    : "
    f"{zero_transaction_before}"
)

print(
    f"Zero transaction cells after     : "
    f"{zero_transaction_after}"
)

print(
    f"Zero count preserved              : "
    f"{zero_count_preserved}"
)

print()

print(
    f"Numeric transaction cells        : "
    f"{numeric_cells}"
)

print(
    f"Non-numeric transaction cells    : "
    f"{non_numeric_cells}"
)

print()

print(
    f"Transaction value mismatches     : "
    f"{transaction_value_mismatches}"
)

print(
    f"Saved-file mismatches             : "
    f"{saved_transaction_mismatches}"
)

print()

print(
    f"Missing values imputed            : False"
)

print(
    f"NaN converted to zero             : False"
)

print(
    f"Data values modified              : False"
)

print(
    f"Negative volume cells             : "
    f"{negative_volume_cells}"
)

print(
    f"Negative value cells              : "
    f"{negative_value_cells}"
)

print(
    f"Positive infinity cells          : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells          : "
    f"{negative_infinity_cells}"
)


# =============================================================================
# SUCCESS
# =============================================================================

print("\n" + "=" * 90)
print("STEP 4 COMPLETE")
print("=" * 90)

print("✓ Input dimensions validated")
print("✓ Transaction columns identified")
print("✓ Structural rows preserved")
print("✓ Embedded header rows preserved")
print("✓ Missing values identified")
print("✓ Missing values preserved")
print("✓ Existing zeros preserved")
print("✓ No missing-value imputation performed")
print("✓ No NaN converted to zero")
print("✓ No transaction values modified")
print("✓ No negative transaction values")
print("✓ No infinity values")
print("✓ Transaction values validated")
print("✓ Saved-file integrity validated")
print("✓ All assertions passed")
print("✓ Checkpoint saved")

print()
print(f"Output file: {OUTPUT_FILE}")
print("=" * 90)

STEP 4: MISSING VALUE ANALYSIS & CONTROLLED CLEANING

INPUT FILE
------------------------------------------------------------------------------------------
Input file : NPCI_Step3_Cleaned.xlsx
Rows       : 40
Columns    : 42
✓ Input dimensions validated

CORE COLUMN VALIDATION
------------------------------------------------------------------------------------------
✓ Sr_No present
✓ NPCI_Operated_System present

TRANSACTION COLUMN IDENTIFICATION
------------------------------------------------------------------------------------------
Transaction columns : 40
Volume columns      : 20
Value columns       : 20
✓ Transaction column structure validated

STRUCTURAL ROW IDENTIFICATION
------------------------------------------------------------------------------------------
Embedded header rows : 2
Embedded headers    : [0, 29]
Empty-system rows    : 2
Empty-system indices : [29, 39]
Structural rows      : 3
Structural indices   : [0, 29, 39]
Transaction rows     : 37
✓ Structural rows iden

In [14]:
# ============================================================================
# STEP 5: DUPLICATE RECORD ANALYSIS & CONTROLLED CLEANING
# ============================================================================
#
# INPUT:
#     NPCI_Step4_MissingValue_Handled.xlsx
#
# OUTPUT:
#     NPCI_Step5_Duplicate_Handled.xlsx
#
# PURPOSE:
#     1. Validate the Step 4 checkpoint
#     2. Identify structural rows
#     3. Detect exact duplicate transaction records
#     4. Remove only exact duplicate transaction rows
#     5. Preserve structural/header rows
#     6. Preserve all transaction values
#     7. Validate the saved output
#
# IMPORTANT:
#     - No missing-value imputation
#     - No NaN-to-zero conversion
#     - No numerical transformations
#     - No negative-value correction
#     - No modification of genuine transaction values
# ============================================================================

import os
import re
import numpy as np
import pandas as pd


# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_FILE = "NPCI_Step4_MissingValue_Handled.xlsx"
OUTPUT_FILE = "NPCI_Step5_Duplicate_Handled.xlsx"

EXPECTED_ROWS = 40
EXPECTED_COLUMNS = 42
EXPECTED_TRANSACTION_COLUMNS = 40
EXPECTED_VOLUME_COLUMNS = 20
EXPECTED_VALUE_COLUMNS = 20


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def is_blank(value):
    """Return True for NaN/None/empty/whitespace-only values."""
    if pd.isna(value):
        return True
    return str(value).strip() == ""


def is_embedded_header_row(row, transaction_columns):
    """
    Detect embedded header rows such as:
        Volume (in Mn)
        Value (in Bn)
    """
    header_values = 0

    for col in transaction_columns:
        value = row[col]

        if pd.isna(value):
            continue

        text = str(value).strip().lower()

        if text in {
            "volume (in mn)",
            "value (in bn)"
        }:
            header_values += 1

    return header_values >= max(2, len(transaction_columns) // 4)


# ============================================================================
# STEP 5 HEADER
# ============================================================================

print("=" * 90)
print("STEP 5: DUPLICATE RECORD ANALYSIS & CONTROLLED CLEANING")
print("=" * 90)


# ============================================================================
# LOAD INPUT
# ============================================================================

print("\nINPUT FILE")
print("-" * 90)
print(f"Input file : {INPUT_FILE}")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}"
    )

df_original = pd.read_excel(INPUT_FILE)

df = df_original.copy(deep=True)

print(f"Rows       : {df.shape[0]}")
print(f"Columns    : {df.shape[1]}")


# ============================================================================
# INPUT DIMENSION VALIDATION
# ============================================================================

assert df.shape[0] == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS} rows, found {df.shape[0]}"
)

assert df.shape[1] == EXPECTED_COLUMNS, (
    f"Expected {EXPECTED_COLUMNS} columns, found {df.shape[1]}"
)

print("✓ Input dimensions validated")


# ============================================================================
# CORE COLUMN VALIDATION
# ============================================================================

print("\nCORE COLUMN VALIDATION")
print("-" * 90)

assert "Sr_No" in df.columns
assert "NPCI_Operated_System" in df.columns

print("✓ Sr_No present")
print("✓ NPCI_Operated_System present")


# ============================================================================
# TRANSACTION COLUMN IDENTIFICATION
# ============================================================================

transaction_columns = [
    col for col in df.columns
    if re.search(r"_(Volume|Value)_in_", str(col))
]

volume_columns = [
    col for col in transaction_columns
    if "_Volume_in_" in str(col)
]

value_columns = [
    col for col in transaction_columns
    if "_Value_in_" in str(col)
]

print("\nTRANSACTION COLUMN IDENTIFICATION")
print("-" * 90)

print(f"Transaction columns : {len(transaction_columns)}")
print(f"Volume columns      : {len(volume_columns)}")
print(f"Value columns       : {len(value_columns)}")

assert len(transaction_columns) == EXPECTED_TRANSACTION_COLUMNS
assert len(volume_columns) == EXPECTED_VOLUME_COLUMNS
assert len(value_columns) == EXPECTED_VALUE_COLUMNS

print("✓ Transaction column structure validated")


# ============================================================================
# ORIGINAL DATA SNAPSHOT
# ============================================================================

df_before = df.copy(deep=True)


# ============================================================================
# STRUCTURAL ROW IDENTIFICATION
# ============================================================================

print("\nSTRUCTURAL ROW IDENTIFICATION")
print("-" * 90)

system_col = "NPCI_Operated_System"

empty_system_indices = [
    idx
    for idx in df.index
    if is_blank(df.loc[idx, system_col])
]

embedded_header_indices = [
    idx
    for idx in df.index
    if is_embedded_header_row(
        df.loc[idx],
        transaction_columns
    )
]

structural_indices = sorted(
    set(empty_system_indices) |
    set(embedded_header_indices)
)

transaction_indices = [
    idx
    for idx in df.index
    if idx not in structural_indices
]

print(f"Embedded header rows : {len(embedded_header_indices)}")
print(f"Embedded headers    : {embedded_header_indices}")

print(f"Empty-system rows    : {len(empty_system_indices)}")
print(f"Empty-system indices : {empty_system_indices}")

print(f"Structural rows      : {len(structural_indices)}")
print(f"Structural indices   : {structural_indices}")

print(f"Transaction rows     : {len(transaction_indices)}")

assert len(transaction_indices) + len(structural_indices) == len(df)

print("✓ Structural rows identified correctly")


# ============================================================================
# STRUCTURAL ROW SNAPSHOT
# ============================================================================

structural_before = (
    df_before.loc[structural_indices].copy(deep=True)
)


# ============================================================================
# TRANSACTION VALUE SNAPSHOT
# ============================================================================

transaction_before = (
    df_before.loc[transaction_indices, transaction_columns]
    .copy(deep=True)
)


# ============================================================================
# MISSING / ZERO COUNTS BEFORE
# ============================================================================

missing_before = int(
    df_before.loc[
        transaction_indices,
        transaction_columns
    ].isna().sum().sum()
)

zero_before = int(
    (
        df_before.loc[
            transaction_indices,
            transaction_columns
        ] == 0
    ).sum().sum()
)

print("\nORIGINAL TRANSACTION STATE")
print("-" * 90)

print(f"Missing transaction cells : {missing_before}")
print(f"Zero transaction cells    : {zero_before}")


# ============================================================================
# DUPLICATE ANALYSIS
# ============================================================================

print("\nDUPLICATE RECORD ANALYSIS")
print("-" * 90)

transaction_df = df.loc[
    transaction_indices
].copy(deep=True)

# Exact duplicate means every column in the transaction row is identical.
duplicate_mask = transaction_df.duplicated(
    keep="first"
)

duplicate_indices = transaction_df.index[
    duplicate_mask
].tolist()

duplicate_count = len(duplicate_indices)

print(f"Exact duplicate transaction rows : {duplicate_count}")

if duplicate_count > 0:
    print(f"Duplicate row indices             : {duplicate_indices}")
else:
    print("✓ No exact duplicate transaction rows detected.")


# ============================================================================
# DUPLICATE KEY ANALYSIS
# ============================================================================
#
# A second check based on the complete row is performed independently.
# This confirms the duplicate detection result.
# ============================================================================

duplicate_key_mask = (
    transaction_df
    .astype(object)
    .apply(
        lambda row: tuple(
            "__NA__" if pd.isna(x) else x
            for x in row
        ),
        axis=1
    )
    .duplicated(keep="first")
)

duplicate_key_indices = transaction_df.index[
    duplicate_key_mask
].tolist()

assert duplicate_indices == duplicate_key_indices

print("✓ Independent duplicate-key validation passed")


# ============================================================================
# CONTROLLED CLEANING
# ============================================================================

print("\nCONTROLLED DUPLICATE HANDLING")
print("-" * 90)

if duplicate_count > 0:

    # Remove ONLY duplicate transaction rows.
    df_cleaned = df.drop(
        index=duplicate_indices
    ).copy(deep=True)

    print(
        f"✓ Removed {duplicate_count} exact duplicate "
        f"transaction row(s)."
    )

else:

    # No duplicate rows: preserve the dataframe exactly.
    df_cleaned = df.copy(deep=True)

    print("✓ No rows removed.")
    print("✓ Input data preserved exactly.")


# ============================================================================
# STRUCTURAL ROW VALIDATION AFTER CLEANING
# ============================================================================

remaining_structural_indices = [
    idx for idx in structural_indices
    if idx in df_cleaned.index
]

assert len(remaining_structural_indices) == len(structural_indices)

structural_after = (
    df_cleaned.loc[
        remaining_structural_indices
    ].copy(deep=True)
)

# Compare structural rows using their original indices.
pd.testing.assert_frame_equal(
    structural_before,
    structural_after,
    check_dtype=False
)

print("✓ Structural rows preserved")


# ============================================================================
# TRANSACTION ROWS AFTER CLEANING
# ============================================================================

remaining_transaction_indices = [
    idx for idx in transaction_indices
    if idx in df_cleaned.index
]

transaction_after = (
    df_cleaned.loc[
        remaining_transaction_indices,
        transaction_columns
    ].copy(deep=True)
)


# ============================================================================
# VALUE INTEGRITY
# ============================================================================
#
# Every remaining transaction row must have exactly the same values as
# before Step 5.
# ============================================================================

original_remaining = (
    transaction_before.loc[
        remaining_transaction_indices
    ]
)

pd.testing.assert_frame_equal(
    original_remaining,
    transaction_after,
    check_dtype=False
)

print("✓ All retained transaction values unchanged")


# ============================================================================
# MISSING VALUE VALIDATION
# ============================================================================

missing_after = int(
    df_cleaned.loc[
        remaining_transaction_indices,
        transaction_columns
    ].isna().sum().sum()
)

zero_after = int(
    (
        df_cleaned.loc[
            remaining_transaction_indices,
            transaction_columns
        ] == 0
    ).sum().sum()
)

print("\nMISSING / ZERO VALIDATION")
print("-" * 90)

print(f"Missing transaction cells before : {missing_before}")
print(f"Missing transaction cells after  : {missing_after}")

print(f"Zero transaction cells before    : {zero_before}")
print(f"Zero transaction cells after     : {zero_after}")

# Duplicate removal should not change values in retained rows.
# Therefore, missing/zero counts are allowed to decrease only if a removed
# duplicate itself contained missing/zero cells.
#
# We validate this explicitly rather than incorrectly demanding equality.

removed_transaction = transaction_before.loc[
    duplicate_indices
]

removed_missing = int(
    removed_transaction.isna().sum().sum()
)

removed_zero = int(
    (removed_transaction == 0).sum().sum()
)

expected_missing_after = missing_before - removed_missing
expected_zero_after = zero_before - removed_zero

assert missing_after == expected_missing_after
assert zero_after == expected_zero_after

print("✓ Missing-value accounting validated")
print("✓ Zero-value accounting validated")


# ============================================================================
# NUMERIC DATA VALIDATION
# ============================================================================

print("\nTRANSACTION DATA TYPE VALIDATION")
print("-" * 90)

transaction_data = df_cleaned.loc[
    remaining_transaction_indices,
    transaction_columns
]

numeric_mask = transaction_data.apply(
    lambda col: pd.to_numeric(
        col,
        errors="coerce"
    ).notna() | col.isna()
)

non_numeric_transaction_cells = int(
    (~numeric_mask).sum().sum()
)

numeric_transaction_cells = int(
    transaction_data.size -
    transaction_data.isna().sum().sum() -
    non_numeric_transaction_cells
)

print(
    f"Numeric transaction data cells    : "
    f"{numeric_transaction_cells}"
)

print(
    f"Non-numeric transaction cells     : "
    f"{non_numeric_transaction_cells}"
)

assert non_numeric_transaction_cells == 0

print("✓ All genuine transaction cells are numeric or missing")


# ============================================================================
# NEGATIVE VALUE CHECK
# ============================================================================

print("\nNEGATIVE VALUE CHECK")
print("-" * 90)

volume_data = transaction_data[volume_columns]
value_data = transaction_data[value_columns]

negative_volume_cells = int(
    (volume_data < 0).sum().sum()
)

negative_value_cells = int(
    (value_data < 0).sum().sum()
)

print(f"Negative volume cells : {negative_volume_cells}")
print(f"Negative value cells  : {negative_value_cells}")

assert negative_volume_cells == 0
assert negative_value_cells == 0

print("✓ No negative transaction values detected")


# ============================================================================
# INFINITY CHECK
# ============================================================================

print("\nINFINITY CHECK")
print("-" * 90)

numeric_transaction_data = transaction_data.apply(
    pd.to_numeric,
    errors="coerce"
)

positive_infinity_cells = int(
    np.isposinf(
        numeric_transaction_data.to_numpy(
            dtype=float
        )
    ).sum()
)

negative_infinity_cells = int(
    np.isneginf(
        numeric_transaction_data.to_numpy(
            dtype=float
        )
    ).sum()
)

print(
    f"Positive infinity cells : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells : "
    f"{negative_infinity_cells}"
)

assert positive_infinity_cells == 0
assert negative_infinity_cells == 0

print("✓ No infinity values detected")


# ============================================================================
# DUPLICATE VALIDATION AFTER CLEANING
# ============================================================================

print("\nDUPLICATE VALIDATION AFTER CLEANING")
print("-" * 90)

clean_transaction_df = df_cleaned.loc[
    remaining_transaction_indices
]

remaining_duplicates = int(
    clean_transaction_df.duplicated(
        keep=False
    ).sum()
)

# Since all exact duplicates were removed, none should remain.
assert remaining_duplicates == 0

print("✓ No exact duplicate transaction records remain")


# ============================================================================
# ROW / COLUMN STRUCTURE VALIDATION
# ============================================================================

print("\nSTRUCTURAL VALIDATION")
print("-" * 90)

assert df_cleaned.shape[1] == EXPECTED_COLUMNS

assert set(df_cleaned.columns) == set(
    df_original.columns
)

print("✓ Column structure preserved")

expected_rows_after = (
    EXPECTED_ROWS - duplicate_count
)

assert df_cleaned.shape[0] == expected_rows_after

print(
    f"✓ Row count validated: "
    f"{df_cleaned.shape[0]}"
)


# ============================================================================
# SAVE OUTPUT
# ============================================================================

print("\nSAVING STEP 5 CHECKPOINT")
print("-" * 90)

df_cleaned.to_excel(
    OUTPUT_FILE,
    index=False
)

assert os.path.exists(OUTPUT_FILE)

print(f"✓ Checkpoint saved: {OUTPUT_FILE}")


# ============================================================================
# RELOAD SAVED FILE
# ============================================================================

df_saved = pd.read_excel(
    OUTPUT_FILE
)


# ============================================================================
# SAVED FILE VALIDATION
# ============================================================================

print("\nSAVED-FILE INTEGRITY VALIDATION")
print("-" * 90)

assert df_saved.shape == df_cleaned.shape

assert list(df_saved.columns) == list(
    df_cleaned.columns
)

print("✓ Saved dimensions validated")
print("✓ Saved column structure validated")


# ============================================================================
# SAVED TRANSACTION VALUE VALIDATION
# ============================================================================

saved_transaction_data = df_saved.loc[
    df_cleaned.index,
    transaction_columns
]

# Because Excel reload can change dtype representations, compare values
# after normalizing numerically where applicable.

for idx in df_cleaned.index:

    for col in transaction_columns:

        original_value = df_cleaned.loc[idx, col]
        saved_value = df_saved.loc[idx, col]

        if pd.isna(original_value) and pd.isna(saved_value):
            continue

        if isinstance(original_value, (int, float, np.number)):
            assert np.isclose(
                float(original_value),
                float(saved_value),
                equal_nan=True
            )
        else:
            assert str(original_value) == str(saved_value)


print("✓ Saved-file transaction values validated")


# ============================================================================
# FINAL COUNTS FROM SAVED FILE
# ============================================================================

saved_transaction_indices = [
    idx
    for idx in df_saved.index
    if not is_blank(
        df_saved.loc[idx, system_col]
    )
    and not is_embedded_header_row(
        df_saved.loc[idx],
        transaction_columns
    )
]

saved_transaction_data = df_saved.loc[
    saved_transaction_indices,
    transaction_columns
]

saved_missing = int(
    saved_transaction_data.isna().sum().sum()
)

saved_zero = int(
    (saved_transaction_data == 0).sum().sum()
)

saved_numeric_mask = saved_transaction_data.apply(
    lambda col: pd.to_numeric(
        col,
        errors="coerce"
    ).notna() | col.isna()
)

saved_non_numeric_cells = int(
    (~saved_numeric_mask).sum().sum()
)


# ============================================================================
# STEP 5 VALIDATION
# ============================================================================

print("\n" + "=" * 90)
print("STEP 5 VALIDATION")
print("=" * 90)

print(
    f"Duplicate transaction rows detected : "
    f"{duplicate_count}"
)

print(
    f"Duplicate transaction rows removed  : "
    f"{duplicate_count}"
)

print(
    f"Rows before                        : "
    f"{df_original.shape[0]}"
)

print(
    f"Rows after                         : "
    f"{df_cleaned.shape[0]}"
)

print(
    f"Columns before                     : "
    f"{df_original.shape[1]}"
)

print(
    f"Columns after                      : "
    f"{df_cleaned.shape[1]}"
)

print(
    f"Missing transaction cells before   : "
    f"{missing_before}"
)

print(
    f"Missing transaction cells after    : "
    f"{missing_after}"
)

print(
    f"Zero transaction cells before      : "
    f"{zero_before}"
)

print(
    f"Zero transaction cells after       : "
    f"{zero_after}"
)

print(
    f"Non-numeric transaction cells      : "
    f"{non_numeric_transaction_cells}"
)

print(
    f"Remaining duplicate rows           : "
    f"{remaining_duplicates}"
)

print(
    f"Negative volume cells              : "
    f"{negative_volume_cells}"
)

print(
    f"Negative value cells               : "
    f"{negative_value_cells}"
)

print(
    f"Positive infinity cells            : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells            : "
    f"{negative_infinity_cells}"
)

print("✓ Structural rows preserved")
print("✓ Retained transaction values unchanged")
print("✓ Missing-value accounting validated")
print("✓ Existing zero-value accounting validated")
print("✓ No negative transaction values")
print("✓ No infinity values")
print("✓ No exact duplicate transaction records remain")
print("✓ Saved-file integrity validated")


# ============================================================================
# FINAL REPORT
# ============================================================================

print("\n" + "=" * 90)
print("STEP 5 FINAL REPORT")
print("=" * 90)

print(f"Input file                       : {INPUT_FILE}")
print(f"Output file                      : {OUTPUT_FILE}")

print(f"\nRows before                      : {df_original.shape[0]}")
print(f"Rows after                       : {df_cleaned.shape[0]}")
print(f"Columns                          : {df_cleaned.shape[1]}")

print(
    f"\nTransaction columns              : "
    f"{len(transaction_columns)}"
)

print(
    f"Volume columns                   : "
    f"{len(volume_columns)}"
)

print(
    f"Value columns                    : "
    f"{len(value_columns)}"
)

print(
    f"\nStructural rows                  : "
    f"{len(structural_indices)}"
)

print(
    f"Transaction rows before          : "
    f"{len(transaction_indices)}"
)

print(
    f"Transaction rows after           : "
    f"{len(remaining_transaction_indices)}"
)

print(
    f"\nExact duplicate rows detected    : "
    f"{duplicate_count}"
)

print(
    f"Exact duplicate rows removed     : "
    f"{duplicate_count}"
)

print(
    f"\nMissing transaction cells before : "
    f"{missing_before}"
)

print(
    f"Missing transaction cells after  : "
    f"{missing_after}"
)

print(
    f"Zero transaction cells before    : "
    f"{zero_before}"
)

print(
    f"Zero transaction cells after     : "
    f"{zero_after}"
)

print(
    f"\nNon-numeric transaction cells    : "
    f"{non_numeric_transaction_cells}"
)

print(
    f"Negative volume cells             : "
    f"{negative_volume_cells}"
)

print(
    f"Negative value cells              : "
    f"{negative_value_cells}"
)

print(
    f"Positive infinity cells           : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells           : "
    f"{negative_infinity_cells}"
)

print(
    "\nTransaction values modified       : False"
)

print(
    "Missing values imputed             : False"
)

print(
    "NaN converted to zero              : False"
)

print(
    "Structural rows modified           : False"
)

print(
    "Saved-file mismatches              : 0"
)


# ============================================================================
# FINAL ASSERTIONS
# ============================================================================

assert df_cleaned.shape[1] == EXPECTED_COLUMNS
assert len(transaction_columns) == EXPECTED_TRANSACTION_COLUMNS
assert len(volume_columns) == EXPECTED_VOLUME_COLUMNS
assert len(value_columns) == EXPECTED_VALUE_COLUMNS

assert non_numeric_transaction_cells == 0

assert negative_volume_cells == 0
assert negative_value_cells == 0

assert positive_infinity_cells == 0
assert negative_infinity_cells == 0

assert remaining_duplicates == 0

assert df_saved.shape == df_cleaned.shape

assert list(df_saved.columns) == list(
    df_cleaned.columns
)

assert saved_missing == missing_after
assert saved_zero == zero_after

assert saved_non_numeric_cells == 0


# ============================================================================
# COMPLETE
# ============================================================================

print("\n" + "=" * 90)
print("STEP 5 COMPLETE")
print("=" * 90)

print("✓ Input dimensions validated")
print("✓ Transaction columns identified")
print("✓ Structural rows identified")
print("✓ Embedded header rows preserved")
print("✓ Duplicate records analyzed")
print("✓ Exact duplicate transaction rows handled")
print("✓ Structural rows preserved")
print("✓ Retained transaction values unchanged")
print("✓ Missing values not imputed")
print("✓ No NaN converted to zero")
print("✓ Existing zero values preserved")
print("✓ No negative transaction values")
print("✓ No infinity values")
print("✓ No duplicate transaction records remain")
print("✓ Saved-file integrity validated")
print("✓ All assertions passed")
print("✓ Checkpoint saved")

print(f"\nOutput file: {OUTPUT_FILE}")

print("=" * 90)

STEP 5: DUPLICATE RECORD ANALYSIS & CONTROLLED CLEANING

INPUT FILE
------------------------------------------------------------------------------------------
Input file : NPCI_Step4_MissingValue_Handled.xlsx
Rows       : 40
Columns    : 42
✓ Input dimensions validated

CORE COLUMN VALIDATION
------------------------------------------------------------------------------------------
✓ Sr_No present
✓ NPCI_Operated_System present

TRANSACTION COLUMN IDENTIFICATION
------------------------------------------------------------------------------------------
Transaction columns : 40
Volume columns      : 20
Value columns       : 20
✓ Transaction column structure validated

STRUCTURAL ROW IDENTIFICATION
------------------------------------------------------------------------------------------
Embedded header rows : 2
Embedded headers    : [0, 29]
Empty-system rows    : 2
Empty-system indices : [29, 39]
Structural rows      : 3
Structural indices   : [0, 29, 39]
Transaction rows     : 37
✓ Stru

In [15]:
# =============================================================================
# STEP 6: DATA CONSISTENCY & INTEGRITY CLEANING
# =============================================================================

import pandas as pd
import numpy as np
import os

# =============================================================================
# CONFIGURATION
# =============================================================================

INPUT_FILE = "NPCI_Step5_Duplicate_Handled.xlsx"
OUTPUT_FILE = "NPCI_Step6_Consistency_Cleaned.xlsx"

EXPECTED_ROWS = 40
EXPECTED_COLUMNS = 42

SYSTEM_COL = "NPCI_Operated_System"
SR_NO_COL = "Sr_No"

# =============================================================================
# STEP 6 HEADER
# =============================================================================

print("=" * 90)
print("STEP 6: DATA CONSISTENCY & INTEGRITY CLEANING")
print("=" * 90)

# =============================================================================
# LOAD INPUT FILE
# =============================================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}"
    )

df = pd.read_excel(INPUT_FILE)

# Keep an untouched copy for validation
df_original = df.copy(deep=True)

# =============================================================================
# INPUT FILE VALIDATION
# =============================================================================

print("\nINPUT FILE")
print("-" * 90)

print(f"Input file : {INPUT_FILE}")
print(f"Rows       : {df.shape[0]}")
print(f"Columns    : {df.shape[1]}")

assert df.shape[0] == EXPECTED_ROWS
assert df.shape[1] == EXPECTED_COLUMNS

print("✓ Input dimensions validated")

# =============================================================================
# CORE COLUMN VALIDATION
# =============================================================================

print("\nCORE COLUMN VALIDATION")
print("-" * 90)

assert SR_NO_COL in df.columns
print("✓ Sr_No present")

assert SYSTEM_COL in df.columns
print("✓ NPCI_Operated_System present")

# =============================================================================
# TRANSACTION COLUMN IDENTIFICATION
# =============================================================================

transaction_columns = [
    col for col in df.columns
    if col.endswith("_Volume_in_Mn")
    or col.endswith("_Value_in_Bn")
]

volume_columns = [
    col for col in transaction_columns
    if col.endswith("_Volume_in_Mn")
]

value_columns = [
    col for col in transaction_columns
    if col.endswith("_Value_in_Bn")
]

print("\nTRANSACTION COLUMN IDENTIFICATION")
print("-" * 90)

print(f"Transaction columns : {len(transaction_columns)}")
print(f"Volume columns      : {len(volume_columns)}")
print(f"Value columns       : {len(value_columns)}")

assert len(transaction_columns) == 40
assert len(volume_columns) == 20
assert len(value_columns) == 20

print("✓ Transaction column structure validated")

# =============================================================================
# STRUCTURAL ROW IDENTIFICATION
# =============================================================================

embedded_header_rows = []

for idx in df.index:

    row_values = df.loc[idx, transaction_columns]

    header_count = row_values.astype(str).isin(
        ["Volume (in Mn)", "Value (in Bn)"]
    ).sum()

    if header_count >= 2:
        embedded_header_rows.append(idx)


empty_system_rows = []

for idx in df.index:

    system_value = df.loc[idx, SYSTEM_COL]

    if pd.isna(system_value):
        empty_system_rows.append(idx)

    elif str(system_value).strip() == "":
        empty_system_rows.append(idx)


structural_rows = sorted(
    set(embedded_header_rows + empty_system_rows)
)

transaction_rows = [
    idx for idx in df.index
    if idx not in structural_rows
]

print("\nSTRUCTURAL ROW IDENTIFICATION")
print("-" * 90)

print(f"Embedded header rows : {len(embedded_header_rows)}")
print(f"Embedded headers     : {embedded_header_rows}")

print(f"Empty-system rows    : {len(empty_system_rows)}")
print(f"Empty-system indices : {empty_system_rows}")

print(f"Structural rows      : {len(structural_rows)}")
print(f"Structural indices   : {structural_rows}")

print(f"Transaction rows     : {len(transaction_rows)}")

assert embedded_header_rows == [0, 29]
assert empty_system_rows == [29, 39]
assert structural_rows == [0, 29, 39]
assert len(transaction_rows) == 37

print("✓ Structural rows identified correctly")

# =============================================================================
# ORIGINAL STATE
# =============================================================================

original_transaction = df.loc[
    transaction_rows,
    transaction_columns
].copy(deep=True)

original_system = df[SYSTEM_COL].copy(deep=True)

original_sr_no = df[SR_NO_COL].copy(deep=True)

missing_before = int(
    original_transaction.isna().sum().sum()
)

zero_before = int(
    (original_transaction == 0).sum().sum()
)

print("\nORIGINAL DATA STATE")
print("-" * 90)

print(f"Missing transaction cells : {missing_before}")
print(f"Zero transaction cells    : {zero_before}")

# =============================================================================
# TEXT CONSISTENCY CHECK
# =============================================================================

print("\nTEXT CONSISTENCY CHECK")
print("-" * 90)

system_whitespace_before = 0

for idx in transaction_rows:

    value = df.loc[idx, SYSTEM_COL]

    if pd.notna(value):

        text_value = str(value)

        if text_value != text_value.strip():
            system_whitespace_before += 1

print(
    f"System names with leading/trailing spaces : "
    f"{system_whitespace_before}"
)

# =============================================================================
# CONTROLLED TEXT CLEANING
# =============================================================================
#
# Only genuine transaction-row system names are cleaned.
#
# Structural rows are NOT modified.
#
# Cleaning performed:
#   1. Remove leading whitespace
#   2. Remove trailing whitespace
#
# Internal spaces and spelling/capitalization are NOT changed because
# automatically changing them could alter legitimate system names.
# =============================================================================

print("\nCONTROLLED TEXT CONSISTENCY CLEANING")
print("-" * 90)

text_changes = 0

for idx in transaction_rows:

    value = df.loc[idx, SYSTEM_COL]

    if pd.notna(value):

        original_value = str(value)
        cleaned_value = original_value.strip()

        if original_value != cleaned_value:

            df.loc[idx, SYSTEM_COL] = cleaned_value
            text_changes += 1

print(
    f"System-name formatting changes : "
    f"{text_changes}"
)

print("✓ Leading/trailing whitespace cleaned where present")
print("✓ Internal text content preserved")
print("✓ Structural rows preserved")

# =============================================================================
# SR_NO CONSISTENCY CHECK
# =============================================================================

print("\nSr_No CONSISTENCY CHECK")
print("-" * 90)

print(f"Sr_No dtype : {df[SR_NO_COL].dtype}")

# Do NOT force Sr_No to integer.
# Preserve its existing representation.

assert df[SR_NO_COL].dtype == df_original[SR_NO_COL].dtype

print("✓ Sr_No dtype preserved")

# =============================================================================
# TRANSACTION NUMERIC VALIDATION
# =============================================================================

print("\nTRANSACTION NUMERIC VALIDATION")
print("-" * 90)

numeric_transaction_cells = 0
non_numeric_transaction_cells = 0

for col in transaction_columns:

    series = df.loc[transaction_rows, col]

    numeric_series = pd.to_numeric(
        series,
        errors="coerce"
    )

    invalid_mask = (
        series.notna()
        & numeric_series.isna()
    )

    invalid_count = int(
        invalid_mask.sum()
    )

    non_numeric_transaction_cells += invalid_count

    numeric_transaction_cells += int(
        numeric_series.notna().sum()
    )

print(
    f"Numeric transaction cells     : "
    f"{numeric_transaction_cells}"
)

print(
    f"Non-numeric transaction cells : "
    f"{non_numeric_transaction_cells}"
)

assert non_numeric_transaction_cells == 0

print("✓ All genuine transaction cells are numeric or missing")

# =============================================================================
# NUMERIC VALUE FORMAT CONSISTENCY
# =============================================================================
#
# Verify that transaction values remain numerically interpretable.
#
# No rounding is performed.
# No scaling is performed.
# No unit conversion is performed.
# =============================================================================

print("\nNUMERIC VALUE FORMAT VALIDATION")
print("-" * 90)

conversion_failures = 0

for col in transaction_columns:

    original_series = df.loc[
        transaction_rows,
        col
    ]

    converted_series = pd.to_numeric(
        original_series,
        errors="coerce"
    )

    invalid = (
        original_series.notna()
        & converted_series.isna()
    )

    conversion_failures += int(
        invalid.sum()
    )

assert conversion_failures == 0

print("✓ All transaction values have valid numeric format")

# =============================================================================
# NEGATIVE VALUE CHECK
# =============================================================================

print("\nNEGATIVE VALUE CHECK")
print("-" * 90)

negative_volume_cells = 0
negative_value_cells = 0

for col in volume_columns:

    numeric_series = pd.to_numeric(
        df.loc[transaction_rows, col],
        errors="coerce"
    )

    negative_volume_cells += int(
        (numeric_series < 0).sum()
    )


for col in value_columns:

    numeric_series = pd.to_numeric(
        df.loc[transaction_rows, col],
        errors="coerce"
    )

    negative_value_cells += int(
        (numeric_series < 0).sum()
    )

print(f"Negative volume cells : {negative_volume_cells}")
print(f"Negative value cells  : {negative_value_cells}")

assert negative_volume_cells == 0
assert negative_value_cells == 0

print("✓ No negative transaction values detected")

# =============================================================================
# INFINITY CHECK
# =============================================================================

print("\nINFINITY CHECK")
print("-" * 90)

positive_infinity_cells = 0
negative_infinity_cells = 0

for col in transaction_columns:

    numeric_series = pd.to_numeric(
        df.loc[transaction_rows, col],
        errors="coerce"
    )

    positive_infinity_cells += int(
        np.isposinf(numeric_series).sum()
    )

    negative_infinity_cells += int(
        np.isneginf(numeric_series).sum()
    )

print(
    f"Positive infinity cells : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells : "
    f"{negative_infinity_cells}"
)

assert positive_infinity_cells == 0
assert negative_infinity_cells == 0

print("✓ No infinity values detected")

# =============================================================================
# MISSING VALUE VALIDATION
# =============================================================================

print("\nMISSING VALUE VALIDATION")
print("-" * 90)

missing_after = int(
    df.loc[
        transaction_rows,
        transaction_columns
    ].isna().sum().sum()
)

zero_after = int(
    (
        df.loc[
            transaction_rows,
            transaction_columns
        ] == 0
    ).sum().sum()
)

print(
    f"Missing transaction cells before : "
    f"{missing_before}"
)

print(
    f"Missing transaction cells after  : "
    f"{missing_after}"
)

print(
    f"Zero transaction cells before    : "
    f"{zero_before}"
)

print(
    f"Zero transaction cells after     : "
    f"{zero_after}"
)

missingness_preserved = (
    missing_before == missing_after
)

zero_count_preserved = (
    zero_before == zero_after
)

assert missingness_preserved
assert zero_count_preserved

print("✓ Missing values preserved")
print("✓ Existing zeros preserved")

# =============================================================================
# TRANSACTION VALUE INTEGRITY
# =============================================================================
#
# IMPORTANT:
# Text formatting may have changed, but transaction values MUST NOT change.
# =============================================================================

transaction_after = df.loc[
    transaction_rows,
    transaction_columns
]

transaction_values_match = (
    original_transaction.equals(
        transaction_after
    )
)

print("\nTRANSACTION VALUE INTEGRITY CHECK")
print("-" * 90)

print(
    f"Transaction values unchanged : "
    f"{transaction_values_match}"
)

assert transaction_values_match

print("✓ All transaction values unchanged")

# =============================================================================
# STRUCTURAL ROW INTEGRITY
# =============================================================================

structural_original = df_original.loc[
    structural_rows
].copy(deep=True)

structural_after = df.loc[
    structural_rows
].copy(deep=True)

structural_rows_unchanged = (
    structural_original.equals(
        structural_after
    )
)

print("\nSTRUCTURAL ROW INTEGRITY CHECK")
print("-" * 90)

assert structural_rows_unchanged

print("✓ Structural rows completely unchanged")

# =============================================================================
# COLUMN STRUCTURE VALIDATION
# =============================================================================

print("\nCOLUMN STRUCTURE VALIDATION")
print("-" * 90)

assert list(df.columns) == list(
    df_original.columns
)

assert df.shape == df_original.shape

print("✓ Column order preserved")
print("✓ Column names preserved")
print("✓ Row count preserved")
print("✓ Column count preserved")

# =============================================================================
# SYSTEM NAME VALIDATION
# =============================================================================

print("\nSYSTEM NAME VALIDATION")
print("-" * 90)

empty_system_after = 0

for idx in transaction_rows:

    value = df.loc[idx, SYSTEM_COL]

    if pd.isna(value):
        empty_system_after += 1

    elif str(value).strip() == "":
        empty_system_after += 1

print(
    f"Empty system names in transaction rows : "
    f"{empty_system_after}"
)

# We do NOT impute missing system names.
# Existing missing values remain missing.

assert empty_system_after == 0

print("✓ All genuine transaction rows have system names")

# =============================================================================
# STRUCTURAL SYSTEM NAME VALIDATION
# =============================================================================

structural_system_before = (
    df_original.loc[
        structural_rows,
        SYSTEM_COL
    ]
)

structural_system_after = (
    df.loc[
        structural_rows,
        SYSTEM_COL
    ]
)

assert structural_system_before.equals(
    structural_system_after
)

print("✓ Structural system-name values preserved")

# =============================================================================
# DATA MODIFICATION SUMMARY
# =============================================================================

print("\nDATA MODIFICATION SUMMARY")
print("-" * 90)

# Transaction data must not change.
transaction_modified = not transaction_values_match

# Structural data must not change.
structural_modified = not structural_rows_unchanged

# Dataset dimensions must not change.
dimensions_modified = (
    df.shape != df_original.shape
)

print(
    f"Transaction data modified : "
    f"{transaction_modified}"
)

print(
    f"Structural rows modified  : "
    f"{structural_modified}"
)

print(
    f"Dimensions modified       : "
    f"{dimensions_modified}"
)

assert transaction_modified is False
assert structural_modified is False
assert dimensions_modified is False

# =============================================================================
# SAVE STEP 6 CHECKPOINT
# =============================================================================

print("\nSAVING STEP 6 CHECKPOINT")
print("-" * 90)

df.to_excel(
    OUTPUT_FILE,
    index=False
)

print(
    f"✓ Checkpoint saved: "
    f"{OUTPUT_FILE}"
)

# =============================================================================
# SAVED FILE VALIDATION
# =============================================================================

print("\nSAVED-FILE INTEGRITY VALIDATION")
print("-" * 90)

saved_df = pd.read_excel(
    OUTPUT_FILE
)

# Dimensions
assert saved_df.shape == df.shape

print("✓ Saved dimensions validated")

# Columns
assert list(saved_df.columns) == list(
    df.columns
)

print("✓ Saved column structure validated")

# Transaction data
saved_transaction = saved_df.loc[
    transaction_rows,
    transaction_columns
]

saved_transaction_match = (
    transaction_after.equals(
        saved_transaction
    )
)

assert saved_transaction_match

print("✓ Saved-file transaction values validated")

# Missing values
saved_missing = int(
    saved_transaction.isna().sum().sum()
)

assert saved_missing == missing_after

print("✓ Saved-file missing values validated")

# Zero values
saved_zero = int(
    (saved_transaction == 0).sum().sum()
)

assert saved_zero == zero_after

print("✓ Saved-file zero values validated")

# Structural rows
saved_structural = saved_df.loc[
    structural_rows
]

assert saved_structural.equals(
    df.loc[structural_rows]
)

print("✓ Saved structural rows validated")

# =============================================================================
# FINAL ASSERTIONS
# =============================================================================

assert len(transaction_columns) == 40
assert len(volume_columns) == 20
assert len(value_columns) == 20

assert len(structural_rows) == 3
assert len(transaction_rows) == 37

assert non_numeric_transaction_cells == 0

assert negative_volume_cells == 0
assert negative_value_cells == 0

assert positive_infinity_cells == 0
assert negative_infinity_cells == 0

assert missingness_preserved
assert zero_count_preserved

assert transaction_values_match
assert structural_rows_unchanged

assert df.shape == df_original.shape
assert list(df.columns) == list(
    df_original.columns
)

assert transaction_modified is False
assert structural_modified is False
assert dimensions_modified is False

assert saved_transaction_match
assert saved_missing == missing_after
assert saved_zero == zero_after

# =============================================================================
# STEP 6 FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("STEP 6 VALIDATION")
print("=" * 90)

print(
    f"Rows before                       : "
    f"{df_original.shape[0]}"
)

print(
    f"Rows after                        : "
    f"{df.shape[0]}"
)

print(
    f"Columns before                    : "
    f"{df_original.shape[1]}"
)

print(
    f"Columns after                     : "
    f"{df.shape[1]}"
)

print(
    f"Transaction columns              : "
    f"{len(transaction_columns)}"
)

print(
    f"Volume columns                   : "
    f"{len(volume_columns)}"
)

print(
    f"Value columns                    : "
    f"{len(value_columns)}"
)

print(
    f"Structural rows                  : "
    f"{len(structural_rows)}"
)

print(
    f"Transaction data rows            : "
    f"{len(transaction_rows)}"
)

print(
    f"System-name formatting changes   : "
    f"{text_changes}"
)

print(
    f"Missing transaction cells before : "
    f"{missing_before}"
)

print(
    f"Missing transaction cells after  : "
    f"{missing_after}"
)

print(
    f"Zero transaction cells before    : "
    f"{zero_before}"
)

print(
    f"Zero transaction cells after     : "
    f"{zero_after}"
)

print(
    f"Numeric transaction cells        : "
    f"{numeric_transaction_cells}"
)

print(
    f"Non-numeric transaction cells    : "
    f"{non_numeric_transaction_cells}"
)

print(
    f"Negative volume cells             : "
    f"{negative_volume_cells}"
)

print(
    f"Negative value cells              : "
    f"{negative_value_cells}"
)

print(
    f"Positive infinity cells           : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells           : "
    f"{negative_infinity_cells}"
)

print(
    f"Transaction values modified       : "
    f"{transaction_modified}"
)

print(
    f"Structural rows modified          : "
    f"{structural_modified}"
)

print(
    f"Missing values imputed             : "
    f"False"
)

print(
    f"NaN converted to zero              : "
    f"False"
)

print(
    f"Saved-file mismatches              : "
    f"{not saved_transaction_match}"
)

# =============================================================================
# FINAL COMPLETION MESSAGE
# =============================================================================

print("\n" + "=" * 90)
print("STEP 6 COMPLETE")
print("=" * 90)

print("✓ Input dimensions validated")
print("✓ Transaction columns identified")
print("✓ Structural rows identified")
print("✓ Embedded header rows preserved")
print("✓ Text consistency checked")
print("✓ Safe text formatting cleaned")
print("✓ Sr_No structure preserved")
print("✓ Transaction numeric format validated")
print("✓ Missing values preserved")
print("✓ Existing zeros preserved")
print("✓ No missing-value imputation performed")
print("✓ No NaN converted to zero")
print("✓ No transaction values modified")
print("✓ No negative transaction values")
print("✓ No infinity values")
print("✓ Structural rows preserved")
print("✓ Column structure preserved")
print("✓ Saved-file integrity validated")
print("✓ All assertions passed")
print("✓ Checkpoint saved")

print(
    f"\nOutput file: {OUTPUT_FILE}"
)

print("=" * 90)

STEP 6: DATA CONSISTENCY & INTEGRITY CLEANING

INPUT FILE
------------------------------------------------------------------------------------------
Input file : NPCI_Step5_Duplicate_Handled.xlsx
Rows       : 40
Columns    : 42
✓ Input dimensions validated

CORE COLUMN VALIDATION
------------------------------------------------------------------------------------------
✓ Sr_No present
✓ NPCI_Operated_System present

TRANSACTION COLUMN IDENTIFICATION
------------------------------------------------------------------------------------------
Transaction columns : 40
Volume columns      : 20
Value columns       : 20
✓ Transaction column structure validated

STRUCTURAL ROW IDENTIFICATION
------------------------------------------------------------------------------------------
Embedded header rows : 2
Embedded headers     : [0, 29]
Empty-system rows    : 2
Empty-system indices : [29, 39]
Structural rows      : 3
Structural indices   : [0, 29, 39]
Transaction rows     : 37
✓ Structural rows 

In [16]:
# =============================================================================
# STEP 7: FINAL DATA CLEANING VALIDATION & CLEAN DATASET EXPORT
# =============================================================================

import os
import pandas as pd
import numpy as np

print("=" * 90)
print("STEP 7: FINAL DATA CLEANING VALIDATION & CLEAN DATASET EXPORT")
print("=" * 90)


# =============================================================================
# 1. FILE PATHS
# =============================================================================

INPUT_FILE = "NPCI_Step6_Consistency_Cleaned.xlsx"
OUTPUT_FILE = "NPCI_Final_Cleaned.xlsx"


# =============================================================================
# 2. LOAD INPUT FILE
# =============================================================================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}"
    )

df = pd.read_excel(INPUT_FILE)

print("\nINPUT FILE")
print("-" * 90)
print(f"Input file : {INPUT_FILE}")
print(f"Rows       : {df.shape[0]}")
print(f"Columns    : {df.shape[1]}")


# =============================================================================
# 3. INPUT DIMENSION VALIDATION
# =============================================================================

EXPECTED_ROWS = 40
EXPECTED_COLUMNS = 42

assert df.shape[0] == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS} rows, found {df.shape[0]}"
)

assert df.shape[1] == EXPECTED_COLUMNS, (
    f"Expected {EXPECTED_COLUMNS} columns, found {df.shape[1]}"
)

print("✓ Input dimensions validated")


# =============================================================================
# 4. CORE COLUMN VALIDATION
# =============================================================================

print("\nCORE COLUMN VALIDATION")
print("-" * 90)

assert "Sr_No" in df.columns
assert "NPCI_Operated_System" in df.columns

print("✓ Sr_No present")
print("✓ NPCI_Operated_System present")


# =============================================================================
# 5. TRANSACTION COLUMN IDENTIFICATION
# =============================================================================

transaction_columns = [
    col for col in df.columns
    if "Volume_in_Mn" in str(col)
    or "Value_in_Bn" in str(col)
]

volume_columns = [
    col for col in transaction_columns
    if "Volume_in_Mn" in str(col)
]

value_columns = [
    col for col in transaction_columns
    if "Value_in_Bn" in str(col)
]

print("\nTRANSACTION COLUMN IDENTIFICATION")
print("-" * 90)
print(f"Transaction columns : {len(transaction_columns)}")
print(f"Volume columns      : {len(volume_columns)}")
print(f"Value columns       : {len(value_columns)}")

assert len(transaction_columns) == 40
assert len(volume_columns) == 20
assert len(value_columns) == 20

print("✓ Transaction column structure validated")


# =============================================================================
# 6. STRUCTURAL ROW IDENTIFICATION
# =============================================================================

system_col = "NPCI_Operated_System"

system_values = df[system_col]

empty_system_mask = (
    system_values.isna()
    | system_values.astype(str).str.strip().eq("")
)

empty_system_indices = df.index[empty_system_mask].tolist()


# -------------------------------------------------------------------------
# Detect embedded header rows
# -------------------------------------------------------------------------

embedded_header_mask = pd.Series(False, index=df.index)

for idx in df.index:

    matches = 0

    for col in transaction_columns:

        value = df.loc[idx, col]

        if isinstance(value, str):

            value_clean = value.strip().lower()

            if (
                value_clean == "volume (in mn)"
                or value_clean == "value (in bn)"
            ):
                matches += 1

    if matches >= 10:
        embedded_header_mask.loc[idx] = True


embedded_header_indices = df.index[
    embedded_header_mask
].tolist()


# -------------------------------------------------------------------------
# Structural rows
# -------------------------------------------------------------------------

structural_mask = (
    empty_system_mask
    | embedded_header_mask
)

structural_indices = df.index[
    structural_mask
].tolist()

transaction_mask = ~structural_mask

transaction_indices = df.index[
    transaction_mask
].tolist()


print("\nSTRUCTURAL ROW IDENTIFICATION")
print("-" * 90)
print(f"Embedded header rows : {len(embedded_header_indices)}")
print(f"Embedded headers     : {embedded_header_indices}")
print(f"Empty-system rows    : {len(empty_system_indices)}")
print(f"Empty-system indices : {empty_system_indices}")
print(f"Structural rows      : {len(structural_indices)}")
print(f"Structural indices   : {structural_indices}")
print(f"Transaction rows     : {len(transaction_indices)}")

assert len(embedded_header_indices) == 2
assert len(empty_system_indices) == 2
assert len(structural_indices) == 3
assert len(transaction_indices) == 37

print("✓ Structural rows identified correctly")


# =============================================================================
# 7. ORIGINAL FINAL CLEANED DATA STATE
# =============================================================================

transaction_before = df.loc[
    transaction_indices,
    transaction_columns
].copy()

missing_before = int(transaction_before.isna().sum().sum())

zero_before = int(
    (transaction_before == 0).sum().sum()
)

print("\nFINAL CLEANED DATA STATE")
print("-" * 90)
print(f"Missing transaction cells : {missing_before}")
print(f"Zero transaction cells    : {zero_before}")


# =============================================================================
# 8. TRANSACTION DATA TYPE VALIDATION
# =============================================================================

numeric_transaction_cells = 0
non_numeric_transaction_cells = 0

non_numeric_details = []

for col in transaction_columns:

    series = df.loc[
        transaction_indices,
        col
    ]

    numeric_mask = pd.to_numeric(
        series,
        errors="coerce"
    ).notna()

    missing_mask = series.isna()

    valid_mask = numeric_mask | missing_mask

    invalid_mask = ~valid_mask

    invalid_count = int(invalid_mask.sum())

    if invalid_count > 0:

        non_numeric_transaction_cells += invalid_count

        for idx in series.index[invalid_mask]:

            non_numeric_details.append(
                (
                    idx,
                    col,
                    series.loc[idx]
                )
            )

    numeric_transaction_cells += int(
        numeric_mask.sum()
    )


print("\nTRANSACTION DATA TYPE VALIDATION")
print("-" * 90)
print(
    f"Numeric transaction data cells    : "
    f"{numeric_transaction_cells}"
)

print(
    f"Non-numeric transaction cells     : "
    f"{non_numeric_transaction_cells}"
)

assert non_numeric_transaction_cells == 0

print("✓ All genuine transaction cells are numeric or missing")


# =============================================================================
# 9. NEGATIVE VALUE CHECK
# =============================================================================

negative_volume_cells = 0
negative_value_cells = 0

for col in volume_columns:

    series = pd.to_numeric(
        df.loc[transaction_indices, col],
        errors="coerce"
    )

    negative_volume_cells += int(
        (series < 0).sum()
    )


for col in value_columns:

    series = pd.to_numeric(
        df.loc[transaction_indices, col],
        errors="coerce"
    )

    negative_value_cells += int(
        (series < 0).sum()
    )


print("\nNEGATIVE VALUE CHECK")
print("-" * 90)
print(f"Negative volume cells : {negative_volume_cells}")
print(f"Negative value cells  : {negative_value_cells}")

assert negative_volume_cells == 0
assert negative_value_cells == 0

print("✓ No negative transaction values detected")


# =============================================================================
# 10. INFINITY CHECK
# =============================================================================

positive_infinity_cells = 0
negative_infinity_cells = 0

for col in transaction_columns:

    numeric_series = pd.to_numeric(
        df.loc[transaction_indices, col],
        errors="coerce"
    )

    positive_infinity_cells += int(
        np.isposinf(numeric_series).sum()
    )

    negative_infinity_cells += int(
        np.isneginf(numeric_series).sum()
    )


print("\nINFINITY CHECK")
print("-" * 90)
print(
    f"Positive infinity cells : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells : "
    f"{negative_infinity_cells}"
)

assert positive_infinity_cells == 0
assert negative_infinity_cells == 0

print("✓ No infinity values detected")


# =============================================================================
# 11. SYSTEM NAME VALIDATION
# =============================================================================

transaction_system_names = df.loc[
    transaction_indices,
    system_col
]

empty_transaction_system_names = (
    transaction_system_names.isna()
    | transaction_system_names.astype(str).str.strip().eq("")
)

empty_transaction_system_count = int(
    empty_transaction_system_names.sum()
)


print("\nSYSTEM NAME VALIDATION")
print("-" * 90)
print(
    "Empty system names in transaction rows : "
    f"{empty_transaction_system_count}"
)

assert empty_transaction_system_count == 0

print("✓ All genuine transaction rows have system names")


# =============================================================================
# 12. DUPLICATE VALIDATION
# =============================================================================

transaction_duplicate_count = int(
    df.loc[
        transaction_indices,
        [system_col] + transaction_columns
    ].duplicated().sum()
)


print("\nDUPLICATE VALIDATION")
print("-" * 90)
print(
    f"Exact duplicate transaction rows : "
    f"{transaction_duplicate_count}"
)

assert transaction_duplicate_count == 0

print("✓ No exact duplicate transaction records remain")


# =============================================================================
# 13. MISSING VALUE VALIDATION
# =============================================================================

transaction_after_validation = df.loc[
    transaction_indices,
    transaction_columns
].copy()

missing_after = int(
    transaction_after_validation.isna().sum().sum()
)

zero_after = int(
    (transaction_after_validation == 0).sum().sum()
)


missingness_preserved = (
    missing_before == missing_after
)

zero_count_preserved = (
    zero_before == zero_after
)


print("\nMISSING / ZERO VALIDATION")
print("-" * 90)
print(
    f"Missing transaction cells before : "
    f"{missing_before}"
)

print(
    f"Missing transaction cells after  : "
    f"{missing_after}"
)

print(
    f"Zero transaction cells before    : "
    f"{zero_before}"
)

print(
    f"Zero transaction cells after     : "
    f"{zero_after}"
)

assert missingness_preserved
assert zero_count_preserved

print("✓ Missing values preserved")
print("✓ Existing zeros preserved")


# =============================================================================
# 14. TRANSACTION VALUE INTEGRITY CHECK
# =============================================================================

transaction_values_match = (
    transaction_before.equals(
        transaction_after_validation
    )
)


print("\nTRANSACTION VALUE INTEGRITY CHECK")
print("-" * 90)

print(
    f"Transaction values unchanged : "
    f"{transaction_values_match}"
)

assert transaction_values_match

print("✓ All transaction values unchanged")


# =============================================================================
# 15. STRUCTURAL ROW INTEGRITY CHECK
# =============================================================================

# Reload original Step 6 file separately to compare structural rows.
original_df = pd.read_excel(INPUT_FILE)

structural_rows_unchanged = True

for idx in structural_indices:

    original_row = original_df.loc[idx]
    current_row = df.loc[idx]

    if not original_row.equals(current_row):
        structural_rows_unchanged = False
        break


print("\nSTRUCTURAL ROW INTEGRITY CHECK")
print("-" * 90)

print(
    f"Structural rows unchanged : "
    f"{structural_rows_unchanged}"
)

assert structural_rows_unchanged

print("✓ Structural rows completely unchanged")


# =============================================================================
# 16. COLUMN STRUCTURE VALIDATION
# =============================================================================

columns_preserved = (
    list(original_df.columns)
    == list(df.columns)
)

rows_preserved = (
    original_df.shape[0]
    == df.shape[0]
)

column_count_preserved = (
    original_df.shape[1]
    == df.shape[1]
)


print("\nCOLUMN / DIMENSION VALIDATION")
print("-" * 90)

print(
    f"Column order preserved : "
    f"{columns_preserved}"
)

print(
    f"Row count preserved    : "
    f"{rows_preserved}"
)

print(
    f"Column count preserved : "
    f"{column_count_preserved}"
)

assert columns_preserved
assert rows_preserved
assert column_count_preserved

print("✓ Column structure preserved")
print("✓ Row count preserved")
print("✓ Column count preserved")


# =============================================================================
# 17. FINAL DATA CLEANING RULE CHECK
# =============================================================================

print("\nFINAL DATA CLEANING RULE CHECK")
print("-" * 90)

# No imputation
imputation_performed = False

# No conversion of missing values to zero
nan_converted_to_zero = False

# No transaction value modifications
data_values_modified = not transaction_values_match

# No structural row modifications
structural_rows_modified = not structural_rows_unchanged


assert imputation_performed is False
assert nan_converted_to_zero is False
assert data_values_modified is False
assert structural_rows_modified is False

print("✓ No missing-value imputation performed")
print("✓ No NaN converted to zero")
print("✓ No transaction values modified")
print("✓ No structural rows modified")


# =============================================================================
# 18. SAVE FINAL CLEANED DATASET
# =============================================================================

print("\nSAVING FINAL CLEAN DATASET")
print("-" * 90)

df.to_excel(
    OUTPUT_FILE,
    index=False
)

assert os.path.exists(OUTPUT_FILE)

print(
    f"✓ Final cleaned dataset saved: "
    f"{OUTPUT_FILE}"
)


# =============================================================================
# 19. RELOAD SAVED FILE
# =============================================================================

saved_df = pd.read_excel(
    OUTPUT_FILE
)


# =============================================================================
# 20. SAVED-FILE DIMENSION VALIDATION
# =============================================================================

assert saved_df.shape == df.shape

assert list(saved_df.columns) == list(
    df.columns
)

print("\nSAVED-FILE INTEGRITY VALIDATION")
print("-" * 90)
print("✓ Saved dimensions validated")
print("✓ Saved column structure validated")


# =============================================================================
# 21. SAVED-FILE TRANSACTION VALUE VALIDATION
# =============================================================================

saved_transaction = saved_df.loc[
    transaction_indices,
    transaction_columns
]

saved_transaction_values_match = (
    transaction_after_validation.equals(
        saved_transaction
    )
)


print(
    f"✓ Saved-file transaction values "
    f"validated : {saved_transaction_values_match}"
)

assert saved_transaction_values_match


# =============================================================================
# 22. SAVED-FILE MISSING VALUE VALIDATION
# =============================================================================

saved_missing = int(
    saved_transaction.isna().sum().sum()
)

saved_zero = int(
    (saved_transaction == 0).sum().sum()
)


saved_missing_preserved = (
    saved_missing == missing_before
)

saved_zero_preserved = (
    saved_zero == zero_before
)


print(
    f"✓ Saved-file missing values "
    f"validated : {saved_missing_preserved}"
)

print(
    f"✓ Saved-file zero values "
    f"validated : {saved_zero_preserved}"
)

assert saved_missing_preserved
assert saved_zero_preserved


# =============================================================================
# 23. SAVED-FILE STRUCTURAL ROW VALIDATION
# =============================================================================

saved_structural_rows_unchanged = True

for idx in structural_indices:

    original_row = df.loc[idx]
    saved_row = saved_df.loc[idx]

    if not original_row.equals(saved_row):

        saved_structural_rows_unchanged = False
        break


print(
    f"✓ Saved structural rows "
    f"validated : {saved_structural_rows_unchanged}"
)

assert saved_structural_rows_unchanged


# =============================================================================
# 24. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("STEP 7 VALIDATION")
print("=" * 90)

print(
    f"Rows before                       : "
    f"{df.shape[0]}"
)

print(
    f"Rows after                        : "
    f"{saved_df.shape[0]}"
)

print(
    f"Columns before                    : "
    f"{df.shape[1]}"
)

print(
    f"Columns after                     : "
    f"{saved_df.shape[1]}"
)

print(
    f"Transaction columns              : "
    f"{len(transaction_columns)}"
)

print(
    f"Volume columns                   : "
    f"{len(volume_columns)}"
)

print(
    f"Value columns                    : "
    f"{len(value_columns)}"
)

print(
    f"Structural rows                  : "
    f"{len(structural_indices)}"
)

print(
    f"Transaction data rows            : "
    f"{len(transaction_indices)}"
)

print(
    f"Missing transaction cells before : "
    f"{missing_before}"
)

print(
    f"Missing transaction cells after  : "
    f"{missing_after}"
)

print(
    f"Zero transaction cells before    : "
    f"{zero_before}"
)

print(
    f"Zero transaction cells after     : "
    f"{zero_after}"
)

print(
    f"Numeric transaction cells        : "
    f"{numeric_transaction_cells}"
)

print(
    f"Non-numeric transaction cells    : "
    f"{non_numeric_transaction_cells}"
)

print(
    f"Exact duplicate transaction rows : "
    f"{transaction_duplicate_count}"
)

print(
    f"Negative volume cells             : "
    f"{negative_volume_cells}"
)

print(
    f"Negative value cells              : "
    f"{negative_value_cells}"
)

print(
    f"Positive infinity cells           : "
    f"{positive_infinity_cells}"
)

print(
    f"Negative infinity cells           : "
    f"{negative_infinity_cells}"
)

print(
    f"Transaction values modified       : "
    f"{data_values_modified}"
)

print(
    f"Structural rows modified          : "
    f"{structural_rows_modified}"
)

print(
    f"Missing values imputed            : "
    f"{imputation_performed}"
)

print(
    f"NaN converted to zero             : "
    f"{nan_converted_to_zero}"
)

print(
    f"Saved-file values unchanged       : "
    f"{saved_transaction_values_match}"
)


# =============================================================================
# 25. FINAL ASSERTIONS
# =============================================================================

assert df.shape == (40, 42)
assert saved_df.shape == (40, 42)

assert len(transaction_columns) == 40
assert len(volume_columns) == 20
assert len(value_columns) == 20

assert len(structural_indices) == 3
assert len(transaction_indices) == 37

assert numeric_transaction_cells == 1262
assert non_numeric_transaction_cells == 0

assert negative_volume_cells == 0
assert negative_value_cells == 0

assert positive_infinity_cells == 0
assert negative_infinity_cells == 0

assert transaction_duplicate_count == 0

assert missingness_preserved
assert zero_count_preserved

assert transaction_values_match
assert structural_rows_unchanged

assert saved_transaction_values_match
assert saved_missing_preserved
assert saved_zero_preserved
assert saved_structural_rows_unchanged

assert imputation_performed is False
assert nan_converted_to_zero is False
assert data_values_modified is False
assert structural_rows_modified is False


# =============================================================================
# 26. COMPLETION MESSAGE
# =============================================================================

print("\n" + "=" * 90)
print("STEP 7 COMPLETE")
print("=" * 90)

print("✓ Input dimensions validated")
print("✓ Transaction columns identified")
print("✓ Structural rows validated")
print("✓ Embedded header rows preserved")
print("✓ Transaction data validated")
print("✓ All genuine transaction cells are numeric or missing")
print("✓ Missing values preserved")
print("✓ Existing zeros preserved")
print("✓ No missing-value imputation performed")
print("✓ No NaN converted to zero")
print("✓ No negative transaction values")
print("✓ No infinity values")
print("✓ No exact duplicate transaction records")
print("✓ Transaction values unchanged")
print("✓ Structural rows unchanged")
print("✓ Column structure preserved")
print("✓ Final dataset successfully exported")
print("✓ Saved-file integrity validated")
print("✓ All assertions passed")
print("✓ Final cleaning checkpoint saved")

print(f"\nFinal cleaned file: {OUTPUT_FILE}")

print("=" * 90)

STEP 7: FINAL DATA CLEANING VALIDATION & CLEAN DATASET EXPORT

INPUT FILE
------------------------------------------------------------------------------------------
Input file : NPCI_Step6_Consistency_Cleaned.xlsx
Rows       : 40
Columns    : 42
✓ Input dimensions validated

CORE COLUMN VALIDATION
------------------------------------------------------------------------------------------
✓ Sr_No present
✓ NPCI_Operated_System present

TRANSACTION COLUMN IDENTIFICATION
------------------------------------------------------------------------------------------
Transaction columns : 40
Volume columns      : 20
Value columns       : 20
✓ Transaction column structure validated

STRUCTURAL ROW IDENTIFICATION
------------------------------------------------------------------------------------------
Embedded header rows : 2
Embedded headers     : [0, 29]
Empty-system rows    : 2
Empty-system indices : [29, 39]
Structural rows      : 3
Structural indices   : [0, 29, 39]
Transaction rows     : 37
